# Introduction to Python for Finance - Part III - DataFrames
Created By: Cordell L. Tanny, CFA, FRM, FDP

Version: 1

Date of last revision: December 10, 2024

This notebook and all code are the intellectual property of Digital Hub Insights LLC. All rights reserved.

Suggested additional resource: https://www.w3schools.com/python

**Pandas and DataFrames**

*Note and recommendation: Very often, you might want to experiment on your own as you go through this notebook. We recommend you save a copy of this notebook before you start adding cells or changing anything. This way you always have a pristine copy to go back to.*

## What is a DataFrame?

A DataFrame is a two-dimensional, tabular data structure in Pandas, similar to an Excel spreadsheet or SQL table. It is the most commonly used object in Pandas and is highly flexible for analyzing and manipulating data.

For financial time series, a DataFrame is often used to organize date-based data such as stock prices, returns, or other financial indicators.


---

**Key Parts of a DataFrame**

A Pandas DataFrame is made up of the following key components:

1. Rows

- Represent individual records or observations.
Indexed by a unique identifier (default is an integer index, but for financial time series, this is typically a DatetimeIndex).

2. Columns

- Represent the different features or variables of the dataset (e.g., stock prices, volumes, returns).
Each column has a name (header) and a specific data type (e.g., float, int, datetime).

3. Index

- A special label for rows that allows quick access and alignment of data.
For financial time series, the index is often a datetime object representing dates and times.
4. Data

- The actual values stored in the DataFrame, organized in rows and columns.
Can contain numeric data (prices, returns), strings (tickers, sectors), or dates (timestamps).



Let's start by creating a simple dataframe ("df") and looking at its structure and elements.

Do not worry too much about creating a df with sample data right now as will come back to that later.
You can examine the documentation for pd.DataFrame() [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html). But this is something you should absolutely know how to do because creating a dataframe from a dictionary will occur very often.

Our first task is to import pandas and get used to the most common parts of a df and common methods.

In [ ]:
# Import pandas as pd (pd is the accespted convention)
import pandas as pd

# We create a sample dataframe by converting a dictionary to a dataframe
# with the built in pd.DataFrame()
data = {
    'AAPL': [150.25, 151.30, 152.10, 149.80, 148.90],
    'MSFT': [299.50, 300.00, 302.20, 298.20, 297.00],
    'GOOGL': [2800.25, 2825.00, 2830.50, 2790.50, 2780.30]
}

# This creates a list of datetime objects.
dates = pd.date_range(start='2023-01-01', periods=5)

# Create DataFrame
df = pd.DataFrame(data, index=dates)

print(df)

              AAPL   MSFT    GOOGL
2023-01-01  150.25  299.5  2800.25
2023-01-02  151.30  300.0  2825.00
2023-01-03  152.10  302.2  2830.50
2023-01-04  149.80  298.2  2790.50
2023-01-05  148.90  297.0  2780.30


We can see that it is a table. But let's breakdown the elements so that you are comfortable with the nomenclature.

1. Rows
- Each row represents the stock prices for a specific date.
- The index (e.g., 2023-01-01) uniquely identifies each row.
2. Columns
- AAPL, MSFT, and GOOGL are the column labels (representing stock tickers).
- Each column contains the prices for the corresponding stock.
3. Index
- The index is a DatetimeIndex (2023-01-01, 2023-01-02, etc.).
- It provides an efficient way to access rows by date.
4. Data
- The numbers (e.g., 150.25, 299.50) are the actual stock prices for each ticker and date.

### What is a DateTime Object?
A datetime object in Python represents a specific date and time, including components like year, month, day, hour, minute, second, and microsecond. Python's datetime module provides tools for working with dates and times, making it easy to handle, manipulate, and format temporal data.

Here’s an example of creating and printing a datetime object:


In [ ]:
from datetime import datetime

# Create a datetime object
dt = datetime(2023, 1, 1, 12, 30, 45)
print(dt)
# Output: 2023-01-01 12:30:45


2023-01-01 12:30:45


**Why Are Datetime Objects Important?**

1. Time-Based Analysis:

- Many datasets (especially financial ones) are indexed or recorded by time. Understanding how to work with time data is essential for analyzing trends and patterns over specific periods.

2. Accuracy in Financial Data:

- Investments rely on precise timestamps for trades, prices, and economic data. Datetime objects allow exact tracking of when events occur.

3. Efficient Filtering and Slicing:

- You can easily filter or slice data based on specific date ranges using datetime objects. For example, extracting all stock prices for a particular year or month.

4. Time Zones:

- Financial markets operate in different time zones. Datetime objects can handle timezone-aware calculations, ensuring consistency across regions.

In [ ]:
# Sample DataFrame of stock prices
data = {'AAPL': [150, 151, 152, 153, 154],
        'MSFT': [299, 300, 301, 302, 303]}
dates = pd.date_range('2023-01-01', periods=5)

df = pd.DataFrame(data, index=dates)

# Filter stock prices for January 3, 2023
print(df.loc['2023-01-03'])  # Note that we don't need to specify the time and minutes!



AAPL    152
MSFT    301
Name: 2023-01-03 00:00:00, dtype: int64


**Creating a Datetime Object with GMT Timezone**

To add timezone information, you can use the pytz library or datetime.timezone. Here’s how to create a datetime object in GMT:

In [ ]:
from datetime import datetime, timezone, timedelta

# Create a timezone-aware datetime object in GMT
gmt_time = datetime(2023, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
print(gmt_time)


2023-01-01 12:00:00+00:00


So, why are we showing you this?

Very often, when you download time series data from different vendors, you might see time zone information in the datetime index. This can cause a lot of problems in your code.

We want you to get in the habit of examining all important information when you download time series data from day 1!

**How to remove the timezone information**

To remove the timezone information, convert the timezone-aware datetime object to a naive datetime object using the .replace() method:

In [ ]:
# Remove timezone information
naive_time = gmt_time.replace(tzinfo=None)
print(naive_time)


2023-01-01 12:00:00


Now, the +00:00 has been removed, and the datetime object is timezone-naive.

**Key Differences: Timezone-Aware vs. Naive**
1. Timezone-Aware:

- Contains timezone information (e.g., +00:00 for GMT).
- Useful for working with data across multiple time zones.
- Example: 2023-01-01 12:00:00+00:00

2. Timezone-Naive:

- Does not include timezone information.
- Easier for basic analysis but risky when working with data from different time zones.
- Example: 2023-01-01 12:00:00



---




### Key Attributes of a DataFrame

#### 1. `.index`: Access the index (e.g., dates for financial time series).

The `.index` attribute of a Pandas DataFrame provides the labels for the rows. In financial time series data, it is typically a DatetimeIndex, which is crucial for analyzing and manipulating data based on time.

In [ ]:
print(df.index)

DatetimeIndex(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04',
               '2023-01-05'],
              dtype='datetime64[ns]', freq='D')


So what do we see in the output and what should you look out for?

1. DatetimeIndex:

- Indicates that the index contains datetime objects.
- Essential for time-based analysis.

2. Values ('2023-01-01', '2023-01-02', ...):

- These are the individual labels for each row, typically representing dates.
dtype='datetime64[ns]':

3. Specifies the data type as 64-bit datetime objects.

4. freq='D':

- Indicates the frequency of the index. In this case, D means daily.
- Frequencies are critical for understanding gaps or irregularities in the data.

Note that this is specific to a datetime index. We will explore what the index will look like for other types of tabular data later. Right now, we want you to get comfortable with timeseries data!

**Why Do We Check `.index`?**
1. To Understand the Structure of the Data:

- The .index tells you what kind of labels are used for rows (e.g., dates, integers).
- For time series, a DatetimeIndex confirms that the data is time-based.

2. For Consistency in Operations:

- Ensures the index aligns with your expectations (e.g., date frequency).
- Misaligned indices can cause errors when merging or slicing data.

3. To Enable Date-Based Operations:

- A DatetimeIndex allows powerful slicing, filtering, and resampling of data based on dates.

So, what should you look for when checking `.index`?

1. Is the index in the correct format (e.g., datetime, strings, etc.)
2. Is the frequency correct (for timeseries)
3. Are the dates in order? -> More on sorting later
4. Are there duplicates?

#### 2. The `.columns` attribute

The .columns attribute of a Pandas DataFrame provides the labels for the columns, representing the variables or features in the dataset.




In [ ]:
data = {'AAPL': [150, 151, 152, 153, 154],
        'MSFT': [299, 300, 301, 302, 303]}
dates = pd.date_range('2023-01-01', periods=5)

df = pd.DataFrame(data, index=dates)

print(df.columns)

Index(['AAPL', 'MSFT'], dtype='object')


**Why Do We Check .columns?**

1. To Understand the Features:

- Identifies what data is stored in each column (e.g., stock tickers, returns, prices).
2. For Data Validation:

- Confirms that all expected variables are present and correctly labeled.
- Helps avoid errors caused by typos or missing columns.

3. For Efficient Data Manipulation:

- Enables dynamic operations, like renaming columns, selecting specific ones, or filtering.

Let's examine the output from the previous cell:

**Key Components:**

1. Column Labels:

- ['AAPL', 'MSFT', 'GOOGL'] are the column names, often representing stock tickers, metrics, or variables.

2. Data Type (dtype='object'):

- Indicates the labels are strings (object type in Pandas).

**When to use `.columns`:**
1. Inspect the Column Names:
- Quickly check what features are available in the dataset.
2. Select specific columns:
- Use column names to slice the DataFrame
- Example: `print(df[['AAPL', 'MSFT]])`
3. Renaming columns
- Rename columns for clarity or standardization:
4. Check for Missing or Extra Columns:
- Validate the presence of required columns:

**What to look for in the output:**

1. Correct labels
2. Unique labels

#### 3. The `.values` attribute
The .values attribute provides the underlying data of the DataFrame as a NumPy array. It contains the actual numbers, strings, or other types stored in the DataFrame.


In [ ]:
print(df.values)

[[150 299]
 [151 300]
 [152 301]
 [153 302]
 [154 303]]


In [ ]:
print(type(df.values))

<class 'numpy.ndarray'>


Uh-oh! It's an array! We haven't seen those yet!

Don't worry about the fact that the output is an array. However it is very important, since many machine learning algorithms will only accept arrays as inputs, not dataframes.

For now, this is what youm should know:

1. Produces an array as an ouput.
2. Allows you to quickly inspect the data from a df without column and index labels.
3. It can also be used to select a subset of data from the dataframe (more on this later)

#### 4. The `.shape` attribute

The .shape attribute provides the dimensions of the DataFrame as a tuple (number of rows, number of columns).


In [ ]:
print(df)
print(df.shape)

            AAPL  MSFT
2023-01-01   150   299
2023-01-02   151   300
2023-01-03   152   301
2023-01-04   153   302
2023-01-05   154   303
(5, 2)


In [ ]:
# But also note that we can use .len to get the number of rows
print(len(df))

5


**Why Do We Check .shape?**

1. To Understand the Dataset Size:
- Confirms how many rows (observations) and columns (features) the dataset contains.
2. For Validation:
- Ensures the dataset matches expected dimensions before processing.
3. To Debug Issues:
- Quickly spot missing or extra rows/columns.

**When to use `.shape`**
1. Check dataset size.
2. Validate after filtering:
- Check the size after applying filters or trasnformations.
3. Handle large datasets:
- Verify dimensions before running memory-intensive operations.


#### Summary of key attributes

These key attributes provide crucial insights into the structure and content of a DataFrame:

- `.index`: Validates the time-based index for financial data.
- `.columns`: Confirms the presence and labeling of variables.
- `.values`: Accesses raw data for advanced operations.
- `.shape`: Ensures the dataset has the correct dimensions.

By checking these attributes, you can efficiently validate, manipulate, and analyze financial datasets. Let me know if you'd like additional examples or extensions!



---



## What is an Attribute? What is a Method?

### What Is an Attribute?
An attribute is a property or characteristic of an object. It provides information about the object and does not require parentheses to access.

Think of it as a descriptor or label for the object.
Accessed directly using the dot notation: object.attribute.

We've already seen examples of attributes in the previous section.

### What Is a Method?

A method is a function associated with an object. It performs an action or operation on the object, often returning a result or modifying the object.

Think of it as an action or behavior the object can perform.
Methods require parentheses, even if they don't take arguments: object.method().

We will turn to methods now, and show you the most important ones you need to get comfortable with right away.


---



## Key DataFrame Methods

We will go through this section by examining a real financial time series.
We will download historical pricing information for Apple for a 20 year period.
This way, as we work through each method, you will gain the practical experience of what to expect.  

### Downloading Data from Yahoo! Finance Using yFinance

[yfinance](https://pypi.org/project/yfinance/) is a Python package specifically created to easilly retrieve information from Yahoo! Finance.
And guess what data type the output will be?
That's right! A dataframe.

Google Colab is equipped with yfinance, so there is no need to install it.

In [ ]:
# import yfinance
import yfinance as yf  # convention

# It is good practice to specify the ticker, start date and end date as variables
ticker = 'AAPL'
start_date = '2005-01-01'  # Notice that we are setting the date as a string and not a datetime object
end_date = '2024-11-30'

# retrieve the prices
df_aapl = yf.download(tickers=ticker, start=start_date, end=end_date)

[*********************100%***********************]  1 of 1 completed


We are going to start by looking at the attributes that we just covered.

In [ ]:
# Examine the index
df_aapl.index

DatetimeIndex(['2005-01-03', '2005-01-04', '2005-01-05', '2005-01-06',
               '2005-01-07', '2005-01-10', '2005-01-11', '2005-01-12',
               '2005-01-13', '2005-01-14',
               ...
               '2024-11-15', '2024-11-18', '2024-11-19', '2024-11-20',
               '2024-11-21', '2024-11-22', '2024-11-25', '2024-11-26',
               '2024-11-27', '2024-11-29'],
              dtype='datetime64[ns]', name='Date', length=5012, freq=None)

We can see:
1. It is in datetime (eventhough we put the dates in as strings)
2. The index name is 'Date'
3. We have 5012 rows
4. The frequency wasn't determined (we will address this later).

Let's confirm the length:

In [ ]:
print(len(df_aapl))

5012


In [ ]:
# Now let's take a look at the shape
print(df_aapl.shape)

(5012, 6)


Notice that we have 6 columns. Let's find out what they are using the `.columns` attribute.

In [ ]:
print(df_aapl.columns)

MultiIndex([('Adj Close', 'AAPL'),
            (    'Close', 'AAPL'),
            (     'High', 'AAPL'),
            (      'Low', 'AAPL'),
            (     'Open', 'AAPL'),
            (   'Volume', 'AAPL')],
           names=['Price', 'Ticker'])


MultiIndex....What is that???

We aren't going to get into MultiIndex just yet. But thnk of it as having two levels of column headings, one level is the ticker and the other is something related to the pricing information for each day.

We will drop the ticker level to make our life easier.
Don't worry about this code yet; just know that it is a result of the way Yahoo! Finance returns the data.

In [ ]:
# Remove the second level of the MultiIndex
df_aapl.columns = df_aapl.columns.get_level_values(0)
df_aapl.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

### Investigating the data

We will start with these four important methods, and look at what they are, and when you should use. These are typically the first things you would do to investigate a dataframe.

1. `.info()`
2. `.describe()`
3. `.head()`
4. `.tail()`


---



#### 1. `.info()`

**What It Does:**
- Provides a concise summary of the DataFrame, including:
 - Index type and range.
 - Number of non-null values in each column.
 - Data types of each column.
 - Memory usage of the DataFrame.

**Why It’s Useful:**
- Helps you understand the structure of the dataset.
- Quickly identifies missing values and data types.
- Useful for large datasets to check memory efficiency.

In [ ]:
df_aapl.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5012 entries, 2005-01-03 to 2024-11-29
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  5012 non-null   float64
 1   Close      5012 non-null   float64
 2   High       5012 non-null   float64
 3   Low        5012 non-null   float64
 4   Open       5012 non-null   float64
 5   Volume     5012 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 274.1 KB


We can see each of the column names as well as their data types. It will also tell us if there are missing values, based on the non-null count. It also provides the date range in the index.
There is a lot of useful information here!

**Key Information to Look For:**
- Non-Null Count: Indicates whether there are missing values.
- Dtypes: Ensures columns have the correct data type (e.g., float64 for prices).
- Memory Usage: Helps optimize memory when working with large datasets.


---



#### 2. `.describe()`
**What It Does:**
- Provides summary statistics for numeric columns in the DataFrame, such as:
 - count, mean, std, min, 25%, 50%, 75%, max.

**Why It’s Useful:**

- Gives a quick snapshot of the distribution of numeric data.
- Identifies outliers or anomalies in the data.
- Useful for assessing the scale and variability of financial metrics.


In [ ]:
df_aapl.describe()

Price,Adj Close,Close,High,Low,Open,Volume
count,5012.000000,5012.000000,5012.000000,5012.000000,5012.000000,5.012000e+03
mean,50.663972,52.443979,52.961847,51.875645,52.406194,3.872847e+08
std,62.161470,62.198987,62.790470,61.535535,62.136976,3.961285e+08
min,0.953359,1.130179,1.159107,1.117857,1.139107,2.404830e+07
25%,6.044693,7.165804,7.248571,7.098304,7.197232,1.010697e+08
50%,22.236845,24.696250,24.978750,24.511250,24.671250,2.300998e+08
75%,63.440518,65.437498,65.781872,64.291250,64.675625,5.585314e+08
max,237.330002,237.330002,237.809998,234.449997,236.479996,3.372970e+09


As you can see, this method provides descriptive statistics for all numeric data columns.

**Key Information to Look For:**
- Mean and Median (50%): Shows the central tendency of the data.
- Min and Max: Helps spot potential outliers.
- Standard Deviation (std): Indicates the variability of prices.

#### 3. `.head()`
**What It Does:**
- Displays the first few rows of the DataFrame (default is 5 rows).

**Why It’s Useful:**
- Provides a quick preview of the dataset.
- Useful for verifying data imports and checking initial rows.

Note: By default, it will show you the first 5 rows. You can specify how many rows to return inside the parentheses:

df_aapl.head(15) would return the first 15 rows

**Key Information to Look For:**
- Correctness of Columns: Ensure columns and their values are aligned.
- Initial Data: Confirm that dates and financial metrics look valid.



---



In [ ]:
df_aapl.head()

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000
2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400
2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600
2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400


In [ ]:
df_aapl.head(15)

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000
2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400
2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600
2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400
2005-01-10,1.038768,1.231429,1.262500,1.212143,1.246964,1725309600
2005-01-11,0.972489,1.152857,1.234821,1.145357,1.218750,2611627200
2005-01-12,0.986047,1.168929,1.176786,1.130357,1.168750,1919702400
2005-01-13,1.051422,1.246429,1.328929,1.245179,1.316250,3164716800




---


#### 4. `.tail()`
**What It Does:**
- Displays the last few rows of the DataFrame (default is 5 rows).
**Why It’s Useful:**
- Useful for examining recent data, especially in time series.
- Helps confirm the end of the dataset is as expected after filtering or transformations.

This behaves in the exact same way as `.head()`

In [ ]:
df_aapl.tail(10)

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2024-11-15,225.000000,225.000000,226.919998,224.270004,226.399994,47923700
2024-11-18,228.020004,228.020004,229.740005,225.169998,225.250000,44686000
2024-11-19,228.279999,228.279999,230.160004,226.660004,226.979996,36211800
2024-11-20,229.000000,229.000000,229.929993,225.889999,228.059998,35169600
2024-11-21,228.520004,228.520004,230.160004,225.710007,228.880005,42108300
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200
2024-11-27,234.929993,234.929993,235.690002,233.809998,234.470001,33498400



---
### Mean, Standard Deviation and Percent Change

These methods allow you to perform calculations on a single column or subset of columns, helping analyze key statistics and trends in financial time series data.

#### 1. `.mean()`

- What It Does: Calculates the mean (average) value for numeric columns.
- Why It’s Useful: Provides the central tendency of a specific column or the entire dataset. And quickly!

In [ ]:
# Find the average values of every column.
# We will store it in a variable.
averages = df_aapl.mean()
print(averages)

Price
Adj Close    5.066397e+01
Close        5.244398e+01
High         5.296185e+01
Low          5.187565e+01
Open         5.240619e+01
Volume       3.872847e+08
dtype: float64


Please take a minute to look at the results here. Is the format of the output what you expected?
We can see that it is a data type float, which is good, and we can see the average values for each column.

But let's do a quick check, since this will introduce a key concept that you must learn when workig with pandas.

In [ ]:
print(type(averages))

<class 'pandas.core.series.Series'>


It's a series????
What is a series?

##### Difference between a dataframe and a series
1. Definition
- Series: A one-dimensional labeled array in Pandas. It is similar to a list or NumPy array but with labels (index) for each value.
- DataFrame: A two-dimensional labeled data structure in Pandas, similar to an Excel spreadsheet or SQL table, consisting of rows and columns.

2. Structure
- Series:
 - Contains a single column of data.
 - Has one axis (index).
 - Can hold any data type (e.g., numeric, strings, datetime).
- DataFrame:
 - Contains multiple rows and columns.
 - Has two axes:
 - Rows: Indexed by default or custom labels.
 - Columns: Labeled for each variable.

 Always remember that a dataframe will have shape (n, m) and a series will have shape (n,).

 This is really important to remember because not all dataframe methods will work on a series and vice-versa. So, you must always pay attention to the format of your output. Normally, you will want your output to be a dataframe.

 Let's take a closer look at the output:

In [ ]:
averages

,0
Price,
Adj Close,5.066397e+01
Close,5.244398e+01
High,5.296185e+01
Low,5.187565e+01
Open,5.240619e+01
Volume,3.872847e+08


Notice that there is no column heading. It just shows a zero.
We can name the column with:

`averages.name = 'AAPL Stats'`

In [ ]:
averages.name = 'AAPL Stats'
averages

,AAPL Stats
Price,
Adj Close,5.066397e+01
Close,5.244398e+01
High,5.296185e+01
Low,5.187565e+01
Open,5.240619e+01
Volume,3.872847e+08


We will come back to working with series another time.
Let's continue with important dataframe methods.

Just remember to pay attention to the data type of the output.

Notice that when you called .mean(), it computed the averages across the rows. Meaning, it applied it a column, and took the average of all rows in that column. This is the same thing as saying "column-wise".

What if you wanted to calculate the mean "row-wise", or using all values in one row for all columns?

Then you would have to do this:

`df_aapl.mean(axis=1)`





##### The axis parameter

The axis parameter in Pandas controls whether operations like .mean(), .sum(), .std(), and others are applied row-wise or column-wise.

**What axis=0 and axis=1 Mean**

1. axis=0 (Default):

- The operation is applied column-wise.
- For each column, Pandas computes the result across all rows.
- Think: "Down the columns."

2. axis=1:

- The operation is applied row-wise.
- For each row, Pandas computes the result across all columns.
- Think: "Across the rows."


In [ ]:
averages_axis_1 = df_aapl.mean(axis=1)
averages_axis_1

,0
Date,
2005-01-03,1.153320e+08
2005-01-04,1.828017e+08
2005-01-05,1.134056e+08
2005-01-06,1.175925e+08
2005-01-07,3.712417e+08
...,...
2024-11-22,6.361574e+06
2024-11-25,1.502566e+07
2024-11-26,7.664562e+06


Look at how different the output is!
We are getting a time series that shows the average of all columns for each date!

This will be useful down the road. For now, we will be dealing with axis=0, which is the default.

**Note that this axis parameter works for most pandas methods.**

---
#### 2. `.std()`

- What It Does: Calculates the standard deviation, a measure of variability in the data.
- Why It’s Useful: Indicates how much prices fluctuate around the mean.

Pandas is so powerful because it handles the most common transformations and calculations for us.
Standard deviation is just one of many examples.

In [ ]:
std = df_aapl.std()
std

,0
Price,
Adj Close,6.216147e+01
Close,6.219899e+01
High,6.279047e+01
Low,6.153553e+01
Open,6.213698e+01
Volume,3.961285e+08


We know that the scientific notation seen in the output isn't ideal, but let's leave it for now.

---

#### 3. `.pct_change()`
- What It Does: Calculates the percentage change between consecutive rows.
- Why It’s Useful: Converts raw prices into returns, which are crucial for financial analysis.

The percent change method is one of the most useful methods for dataframes and you will be using it quite often.
The beautiful thing about it, is that you can specify the period over which to measure the percent change within the brackets!
By default, it will calculate the percent change of all columns in relation to the previous row.

If, for example, you wanted to measure the percent change vs. 3 periods ago, you would do:

`.pct_change(3)`

In [ ]:
df_pct_change = df_aapl.pct_change()

df_pct_change

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-04,0.010271,0.010270,0.005529,0.005910,-0.015283,0.585004
2005-01-05,0.008758,0.008758,-0.003360,0.017151,0.010503,-0.379625
2005-01-06,0.000775,0.000775,-0.005211,-0.011241,0.003258,0.036920
2005-01-07,0.072811,0.072811,0.072716,0.022422,0.005103,2.157018
...,...,...,...,...,...,...
2024-11-22,0.005908,0.005908,0.002433,0.010412,-0.003583,-0.093568
2024-11-25,0.013051,0.013051,0.010966,0.007367,0.014908,1.361981
2024-11-26,0.009404,0.009404,0.009946,0.015626,0.008079,-0.489908


You can see that each value now shows the percent change compared to the previous row, and in decimal format (1% = 0.01)

You should also notice a couple of things from the output:
1. That when you try to output an entire dataframe, it will show you the first 5 rows, then ... and then the last 5 rows.
2. The first row is now NaN (not a number) because it cannot calculate the percent change on the first row!

Get used to seeing NaN! We will have a whole section dedicated to dealing with NaN.

What do you think the output will look like when we do pct_change(3)?

In [ ]:
df_pct_change_3 = df_aapl.pct_change(3)
df_pct_change_3

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-05,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-06,0.019908,0.019908,-0.003072,0.011662,-0.001699,0.019600
2005-01-07,0.083046,0.083046,0.063541,0.028268,0.018968,1.030844
...,...,...,...,...,...,...
2024-11-22,0.006965,0.006965,0.002433,0.006177,0.004758,0.054029
2024-11-25,0.016900,0.016900,0.014439,0.017044,0.014908,1.563373
2024-11-26,0.028619,0.028619,0.023505,0.033760,0.019442,0.092093


Notice that now we have 3 rows of NaN!
That should not be a surprise since we are doing a percent change vs. the value from 3 rows up.



---



## Working With Subsections of a DataFrame

Thus far, we have worked with the entire dataframe. But often, you might only want to work with specific columns or specific date ranges.
For example, maybe you don't care about doing a percent change on the volume column and only wanted to do it on the Adj Close column.

Let's start with these cases.




### 1. Isolating One Column

Let's say we wanted to work with only the adjuatewd closing price column. To do this, we would enclose the target column name in a list next to the dataframe name.

In [ ]:
adj_close_mean = df_aapl['Adj Close'].mean()
print(adj_close_mean)
print(type(adj_close_mean))

50.66397201126562
<class 'numpy.float64'>


Notice how it comes out! We get just the answer as a float, not as a dataframe or a series.

What if you wanted it as a dataframe or a series?

Enclosing the column label in 2 sets of square brackets will return a series. We can then convert this series to a dataframe.

In [ ]:
df_adj_close_mean = df_aapl[['Adj Close']].mean()
df_adj_close_mean

,0
Price,
Adj Close,50.663972


In [ ]:
# Convert a series to a dataframe
df_adj_close_mean = df_adj_close_mean.to_frame(name='AAPL Mean')
df_adj_close_mean

,AAPL Mean
Price,
Adj Close,50.663972


the method `.to_frame()` will convert a series to a dataframe.
You can use the name parameter in the parentheses so that there will be a column label and not just 0.

### 2. Selecting Multiple Columns
In this case, we want to isolate the Close and Adj Close columns.
We will use a similar syntax.


In [ ]:
# 1) Isolate the columns of interest
df_subset = df_aapl[['Close', 'Adj Close']]
df_subset
# You will see the output has our two columns!
# Note that the column names you want to isolate are enclosed in separate sets of quotes

Price,Close,Adj Close
Date,,
2005-01-03,1.130179,0.953359
2005-01-04,1.141786,0.963151
2005-01-05,1.151786,0.971586
2005-01-06,1.152679,0.972339
2005-01-07,1.236607,1.043136
...,...,...
2024-11-22,229.869995,229.869995
2024-11-25,232.869995,232.869995
2024-11-26,235.059998,235.059998


In [ ]:
# 2) Apply a method
df_subset_mean = df_subset.mean()
df_subset_mean

,0
Price,
Close,52.443979
Adj Close,50.663972


Note that the output moves the columns as rows and returns a series.
You could convert this to a dataframe if you wanted to.

This works for all other methods.

Let's try the pct_change

In [ ]:
df_subset_pct_change = df_subset.pct_change()
df_subset_pct_change

Price,Close,Adj Close
Date,,
2005-01-03,NaN,NaN
2005-01-04,0.010270,0.010271
2005-01-05,0.008758,0.008758
2005-01-06,0.000775,0.000775
2005-01-07,0.072811,0.072811
...,...,...
2024-11-22,0.005908,0.005908
2024-11-25,0.013051,0.013051
2024-11-26,0.009404,0.009404


This returns a dataframe as expected.
But what if you wanted to add the percent change of the Adj Close as a new column instead of a new dataframe?

### Adding a New Column

To add the pct_change of the Adj Close price, you assign the method to the column of interest. In this case, we will calculate the daily return of AAPL and add it as a new column called 'Daily % Change'
Pay attention to the syntax.

In [ ]:
df_aapl['Daily % Change'] = df_aapl['Adj Close'].pct_change()
df_aapl

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN
2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271
2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758
2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811
...,...,...,...,...,...,...,...
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300,0.005908
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800,0.013051
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200,0.009404


Check df_aapl.info() to see the changes

In [ ]:
df_aapl.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5012 entries, 2005-01-03 to 2024-11-29
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Adj Close       5012 non-null   float64
 1   Close           5012 non-null   float64
 2   High            5012 non-null   float64
 3   Low             5012 non-null   float64
 4   Open            5012 non-null   float64
 5   Volume          5012 non-null   int64  
 6   Daily % Change  5011 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 313.2 KB


This is a great example to show how to not only add a column, but to inspect the results afterwardws with .info()!
Notice that we have the new column listed at the end, but that also, we are missing a value, or it shows we have an NaN?
Look at the output and figure out how we were able to tell that!

## Method Chaining

Method chaining is a great example of how amazingly simple Python can be.
Let's say we wanted to calculate the mean daily % change of the Adj Close column.

In [ ]:
mean_pct_change = df_aapl['Adj Close'].pct_change().mean()
mean_pct_change

0.001307320083615373

And that's it! You just did one line of code instead of two.
And you can chain as many as you need to (until something breaks of course). Just remember that order matters, and you can always put things in parentheses if you need to.

---


## Other Important Methods

We will now look at some other important methods:
1. `min()` and `max()`
2. `round()`
3. `argmax()` and `argmin()`
4. `idxmax()` and `idxmin()`


### 1. `max()` and `min()`

**What They Do:**

- min(): Returns the minimum value in the DataFrame or Series.
- max(): Returns the maximum value in the DataFrame or Series.

**Why They’re Useful:**

- Helps find the lowest and highest stock prices, returns, or other metrics.
- Essential for identifying extreme values (e.g., yearly lows or highs).

In [ ]:
print('Maximum Values:')
print(df_aapl.max())
print('\nMinimum Values:')
print(df_aapl.min())

Maximum Values:
Price
Adj Close         2.373300e+02
Close             2.373300e+02
High              2.378100e+02
Low               2.344500e+02
Open              2.364800e+02
Volume            3.372970e+09
Daily % Change    1.390492e-01
dtype: float64

Minimum Values:
Price
Adj Close         9.533592e-01
Close             1.130179e+00
High              1.159107e+00
Low               1.117857e+00
Open              1.139107e+00
Volume            2.404830e+07
Daily % Change   -1.791952e-01
dtype: float64


### 2. `round()`

**What It Does:**
- Rounds numerical values to a specified number of decimal places.

**Why It’s Useful:**
- Ensures consistent formatting for financial data (e.g., prices to 2 decimal places).
- Pandas makes it very easy to round! You just need to put the number of decimal places inside the parentheses.


In [ ]:
# Round to 2 decimal places
print("Rounded values:\n", df_aapl.head().round(2))

# Notice the method chaining!

Rounded values:
 Price       Adj Close  Close  High   Low  Open      Volume  Daily % Change
Date                                                                      
2005-01-03       0.95   1.13  1.16  1.12  1.16   691992000             NaN
2005-01-04       0.96   1.14  1.17  1.12  1.14  1096810400            0.01
2005-01-05       0.97   1.15  1.17  1.14  1.15   680433600            0.01
2005-01-06       0.97   1.15  1.16  1.13  1.15   705555200            0.00
2005-01-07       1.04   1.24  1.24  1.16  1.16  2227450400            0.07


### 3. `argmax()` and `argmin()`

**What They Do:**
- argmax(): Returns the position (index) of the maximum value in a Series.
- argmin(): Returns the position (index) of the minimum value in a Series.

**Why They’re Useful:**
- Identifies where the highest or lowest values occur.
- Helps find peaks and troughs in time series data.


In [ ]:
# Find the index position of the highest and lowest adjusted closing prices
print("Position of max value in AAPL:", df_aapl['Adj Close'].argmax())
print("Position of min value in AAPL:", df_aapl['Adj Close'].argmin())


Position of max value in AAPL: 5011
Position of min value in AAPL: 0


The output here is simply telling you index position 0 (row 0) has the lowest price while index position 5011 (row 5011, starting at 0) has the higest value.

But what if we wanted the datetime index values that correspond to those integer locations?

### 4. `idxmax()` and `idxmin()`

**What They Do:**
- idxmax(): Returns the index label of the maximum value.
- idxmin(): Returns the index label of the minimum value.

**Why They’re Useful:**
- Pinpoints the date or identifier associated with extreme values in time series data.


In [ ]:
print(f"Date of lowest price: {df_aapl['Adj Close'].idxmin()}")
print(f"Date of higest price: {df_aapl['Adj Close'].idxmax()}")

Date of lowest price: 2005-01-03 00:00:00
Date of higest price: 2024-11-29 00:00:00




---



## Using .iloc, .loc and .at for DataFrame Slicing

*This is one of the most important topics in this course, and should be practiced often!*

**Consider this section to be a more powerful Excel equivalent of VLOOKUP.**


Efficiently slicing and accessing data is a cornerstone of financial data analysis. Financial datasets, such as stock prices, returns, and volumes, often span large time periods and involve multiple variables. The ability to extract specific subsets of data—whether by position, label, or exact value—allows analysts to focus on relevant information, perform precise calculations, and identify key trends. Methods like .iloc, .loc, and .at empower users to navigate complex datasets with ease, making it possible to filter by date ranges, isolate specific securities, and retrieve individual values. Mastering these tools is essential for tasks such as portfolio analysis, strategy backtesting, and financial reporting, where accuracy and efficiency are paramount.








### 1. `.iloc`
**What It Does**
- `.iloc` is used for integer-based indexing in a DataFrame.
- Allows you to select rows and columns by their numerical positions.

**How to Use It**
1. Select Specific Rows and Columns:
- df.iloc[row, column]
2. Slice Rows or Columns:
- Use : for slicing, e.g., df.iloc[start:stop, :].


In [ ]:
# First row and all columns; notice that it produces a series (we only have one row...)
print(df_aapl.iloc[0, :])
print(type(df_aapl.iloc[0, :]))

Price
Adj Close         9.533592e-01
Close             1.130179e+00
High              1.162679e+00
Low               1.117857e+00
Open              1.156786e+00
Volume            6.919920e+08
Daily % Change             NaN
Name: 2005-01-03 00:00:00, dtype: float64
<class 'pandas.core.series.Series'>


In [ ]:
# First 5 rows of Adj Close
print(df_aapl.iloc[0:5, 0])


Date
2005-01-03    0.953359
2005-01-04    0.963151
2005-01-05    0.971586
2005-01-06    0.972339
2005-01-07    1.043136
Name: Adj Close, dtype: float64


In [ ]:
# Last 5 rows
print(df_aapl.iloc[-5:, :])


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2024-11-22  229.869995  229.869995  230.720001  228.059998  228.059998   
2024-11-25  232.869995  232.869995  233.250000  229.740005  231.460007   
2024-11-26  235.059998  235.059998  235.570007  233.330002  233.330002   
2024-11-27  234.929993  234.929993  235.690002  233.809998  234.470001   
2024-11-29  237.330002  237.330002  237.809998  233.970001  234.809998   

Price         Volume  Daily % Change  
Date                                  
2024-11-22  38168300        0.005908  
2024-11-25  90152800        0.013051  
2024-11-26  45986200        0.009404  
2024-11-27  33498400       -0.000553  
2024-11-29  28481400        0.010216  


In [ ]:
# Last 5 rows, columns 2 and 3 - Remember 0 indexing!
print(df_aapl.iloc[-5:, 1:3])


Price            Close        High
Date                              
2024-11-22  229.869995  230.720001
2024-11-25  232.869995  233.250000
2024-11-26  235.059998  235.570007
2024-11-27  234.929993  235.690002
2024-11-29  237.330002  237.809998


### 2. `.loc`: Label Based Indexing

**What It Does**
- .loc is used for label-based indexing in a DataFrame.
- Allows you to select rows and columns by their labels (e.g., dates or column names).

**How to Use It**
1. Select Specific Rows and Columns by Label:
- df.loc[row_label, column_label]
2. Slice Rows and Columns by Label:
- df.loc[start_row:end_row, start_col:end_col]

**Why It’s Useful**
- Works naturally with labeled indices like dates or named columns.
- Ideal for slicing time series data or filtering specific columns.

In [ ]:
# Data for a specific date
print(df_aapl.loc['2023-01-03'])


Price
Adj Close         1.237685e+02
Close             1.250700e+02
High              1.309000e+02
Low               1.241700e+02
Open              1.302800e+02
Volume            1.121175e+08
Daily % Change   -3.740459e-02
Name: 2023-01-03 00:00:00, dtype: float64


In [ ]:
# Data for January 2023
print(df_aapl.loc['2023-01-01':'2023-01-31'])


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2023-01-03  123.768463  125.070000  130.899994  124.169998  130.279999   
2023-01-04  125.045029  126.360001  128.660004  125.080002  126.889999   
2023-01-05  123.718971  125.019997  127.769997  124.760002  127.129997   
2023-01-06  128.271118  129.619995  130.289993  124.889999  126.010002   
2023-01-09  128.795578  130.149994  133.410004  129.889999  130.470001   
2023-01-10  129.369553  130.729996  131.259995  128.119995  130.259995   
2023-01-11  132.100830  133.490005  133.509995  130.460007  131.250000   
2023-01-12  132.021652  133.410004  134.259995  131.440002  133.880005   
2023-01-13  133.357620  134.759995  134.919998  131.660004  132.029999   
2023-01-17  134.525360  135.940002  137.289993  134.130005  134.830002   
2023-01-18  133.802917  135.210007  138.610001  135.029999  136.820007   
2023-01-19  133.862320  135.270004  13

In [ ]:
# Slices can be saved as new dataframes
df_aapl_jan = df_aapl.loc['2023-01-01':'2023-01-31']
df_aapl_jan.head()

# Notice how we didn't specify columns here, so it returns all of them

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2023-01-03,123.768463,125.070000,130.899994,124.169998,130.279999,112117500,-0.037405
2023-01-04,125.045029,126.360001,128.660004,125.080002,126.889999,89113600,0.010314
2023-01-05,123.718971,125.019997,127.769997,124.760002,127.129997,80962700,-0.010605
2023-01-06,128.271118,129.619995,130.289993,124.889999,126.010002,87754700,0.036794
2023-01-09,128.795578,130.149994,133.410004,129.889999,130.470001,70790800,0.004089


In [ ]:
# Keeping only specific columns by labels; notice the column labels are in a list.
df_aapl_jan = df_aapl.loc['2023-01-01':'2023-01-31', ['Adj Close', 'Volume']]
df_aapl_jan.head()

Price,Adj Close,Volume
Date,,
2023-01-03,123.768463,112117500
2023-01-04,125.045029,89113600
2023-01-05,123.718971,80962700
2023-01-06,128.271118,87754700
2023-01-09,128.795578,70790800


### 3. `.at`

**What It Does**

- .at is used for fast access to a single scalar value by label.
- It is optimized for quick retrieval and is faster than .loc for single values.

**How to Use It**
1. Access a Single Value:
 - df.at[row_label, column_label]

**Why It’s Useful**
- Highly efficient for retrieving a single value.
- Best used when performance is critical for single value lookups.

**One of the main advantages of using `.at` is that it returns the value and not a dataframe or a series!
Using `.loc` or `.iloc` can return either a dataframe or a series, and this could cause errors in functions when you are trying to access a single value. To avoid ambiguity and uncertainty of what `.iloc` and `.loc` will return when all you want is a single value, use `.at`**

In [ ]:
# Adjusted close price on a specific date
print(df_aapl.at['2023-01-03', 'Adj Close'])


123.76846313476562



---
## Filterin DataFrames Based on Conditions

Filtering DataFrames is a fundamental skill for financial analysis, allowing you to extract and work with subsets of data that meet specific criteria. For example, you might want to isolate days where a stock's adjusted closing price exceeded a certain value or identify high-volume trading days. By defining conditions such as >, <, >=, and combining multiple conditions with logical operators (&, |, ~), you can quickly narrow your focus to the most relevant data for your analysis.

At the heart of filtering is the concept of a Boolean mask. A Boolean mask is a series of True or False values generated by evaluating a condition on a DataFrame or Series. This mask is used to filter rows where the condition is True, effectively creating a new subset of data. Boolean masks are crucial because they make the filtering process efficient and intuitive, allowing you to apply conditions directly to your data without requiring complex loops or manual selection.

Whether you're identifying trends, evaluating trading strategies, or preparing data for visualization, mastering filtering ensures that you can work with clean, targeted datasets tailored to your needs. This section will demonstrate how to filter data based on single and multiple conditions, and how to refine the results to include only specific columns.


### What is a Boolean Mask?

The best way to explain the mask is to demonstrate how it works.

Let's say you want to extract all days from df_aapl when the Daily % Change is greater than 1%.

You would start with this:

In [ ]:
# Boolean mask, all days where Daily % Change is greater than 1%
df_aapl['Daily % Change'] > 0.01

,Daily % Change
Date,
2005-01-03,False
2005-01-04,True
2005-01-05,False
2005-01-06,False
2005-01-07,True
...,...
2024-11-22,False
2024-11-25,True
2024-11-26,False


Notice that it produces a dataframe that just returned the Daily % Change column, but with no values. It just tells you if each row satisfied the condition. But what if you want to retaill ALL columns and ALL values?

### 1. Filtering Rows Based on a Single Condition

Let's look at the example of filtering all days where the Adj Close is > 150.

In [ ]:
# Filter rows where Adj Close is greater than 150
filtered_df = df_aapl[df_aapl['Adj Close'] > 150]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2021-08-30  150.432648  153.119995  153.490005  148.610001  149.000000   
2021-09-02  150.953339  153.649994  154.720001  152.399994  153.869995   
2021-09-03  151.591934  154.300003  154.630005  153.089996  153.759995   
2021-09-07  153.940033  156.690002  157.259995  154.389999  154.970001   
2021-09-08  152.387726  155.110001  157.039993  153.979996  156.979996   
...                ...         ...         ...         ...         ...   
2024-11-22  229.869995  229.869995  230.720001  228.059998  228.059998   
2024-11-25  232.869995  232.869995  233.250000  229.740005  231.460007   
2024-11-26  235.059998  235.059998  235.570007  233.330002  233.330002   
2024-11-27  234.929993  234.929993  235.690002  233.809998  234.470001   
2024-11-29  237.330002  237.330002  237.809998  233.970001  234.809998   

Price         Volume  Daily % Change 

Explanation:

- The condition df_aapl['Adj Close'] > 150 generates a boolean mask (True or False for each row).

- Passing this mask to df_aapl returns only rows where the condition is True.

**Key Point:**
Please remember the syntax for when you want to filter:

`df[df[column name] condition]`

Note that the df name appears twice! Once to create the boolean mask, the second time to retain retrieve the data from the df based on the boolean mask.

### 2. Filtering Rows Where Adj Close >= 150

Let's do the same thing but use the >= operator

In [ ]:
# Filter rows where Adj Close is greater than or equal to 150
filtered_df = df_aapl[df_aapl['Adj Close'] >= 150]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2021-08-30  150.432648  153.119995  153.490005  148.610001  149.000000   
2021-09-02  150.953339  153.649994  154.720001  152.399994  153.869995   
2021-09-03  151.591934  154.300003  154.630005  153.089996  153.759995   
2021-09-07  153.940033  156.690002  157.259995  154.389999  154.970001   
2021-09-08  152.387726  155.110001  157.039993  153.979996  156.979996   
...                ...         ...         ...         ...         ...   
2024-11-22  229.869995  229.869995  230.720001  228.059998  228.059998   
2024-11-25  232.869995  232.869995  233.250000  229.740005  231.460007   
2024-11-26  235.059998  235.059998  235.570007  233.330002  233.330002   
2024-11-27  234.929993  234.929993  235.690002  233.809998  234.470001   
2024-11-29  237.330002  237.330002  237.809998  233.970001  234.809998   

Price         Volume  Daily % Change 

The output is the same since there were no days where the Adj Close was exactly 150.

Let's try filtering on a different column:

### 3. Filtering Based on Another Column


In [ ]:
# Filter rows where Volume is greater than 1,000,000
filtered_df = df_aapl[df_aapl['Volume'] > 1_000_000]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2005-01-03    0.953359    1.130179    1.162679    1.117857    1.156786   
2005-01-04    0.963151    1.141786    1.169107    1.124464    1.139107   
2005-01-05    0.971586    1.151786    1.165179    1.143750    1.151071   
2005-01-06    0.972339    1.152679    1.159107    1.130893    1.154821   
2005-01-07    1.043136    1.236607    1.243393    1.156250    1.160714   
...                ...         ...         ...         ...         ...   
2024-11-22  229.869995  229.869995  230.720001  228.059998  228.059998   
2024-11-25  232.869995  232.869995  233.250000  229.740005  231.460007   
2024-11-26  235.059998  235.059998  235.570007  233.330002  233.330002   
2024-11-27  234.929993  234.929993  235.690002  233.809998  234.470001   
2024-11-29  237.330002  237.330002  237.809998  233.970001  234.809998   

Price           Volume  Daily % Chang

Nothing that different here, as the process is the same.

### 4. Filtering with Multiple Conditions

**Using & (AND Condition):**

Filter rows where Adj Close > 150 and Volume > 1,000,000:

In [ ]:
# Filter rows with multiple conditions
filtered_df = df_aapl[(df_aapl['Adj Close'] > 150) & (df_aapl['Volume'] > 1_000_000)]
print(filtered_df)

# Again, notice that each condition is enclosed in separate parentheses!

Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2021-08-30  150.432648  153.119995  153.490005  148.610001  149.000000   
2021-09-02  150.953339  153.649994  154.720001  152.399994  153.869995   
2021-09-03  151.591934  154.300003  154.630005  153.089996  153.759995   
2021-09-07  153.940033  156.690002  157.259995  154.389999  154.970001   
2021-09-08  152.387726  155.110001  157.039993  153.979996  156.979996   
...                ...         ...         ...         ...         ...   
2024-11-22  229.869995  229.869995  230.720001  228.059998  228.059998   
2024-11-25  232.869995  232.869995  233.250000  229.740005  231.460007   
2024-11-26  235.059998  235.059998  235.570007  233.330002  233.330002   
2024-11-27  234.929993  234.929993  235.690002  233.809998  234.470001   
2024-11-29  237.330002  237.330002  237.809998  233.970001  234.809998   

Price         Volume  Daily % Change 

Explanation:

- Wrap each condition in parentheses () to avoid errors.
- Combine conditions with & for "AND" logic.

**Using | (OR Condition):**

Filter rows where Adj Close > 150 or Volume > 1,000,000:

In [ ]:
# Filter rows with OR condition
filtered_df = df_aapl[(df_aapl['Adj Close'] > 150) | (df_aapl['Volume'] > 1_000_000)]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2005-01-03    0.953359    1.130179    1.162679    1.117857    1.156786   
2005-01-04    0.963151    1.141786    1.169107    1.124464    1.139107   
2005-01-05    0.971586    1.151786    1.165179    1.143750    1.151071   
2005-01-06    0.972339    1.152679    1.159107    1.130893    1.154821   
2005-01-07    1.043136    1.236607    1.243393    1.156250    1.160714   
...                ...         ...         ...         ...         ...   
2024-11-22  229.869995  229.869995  230.720001  228.059998  228.059998   
2024-11-25  232.869995  232.869995  233.250000  229.740005  231.460007   
2024-11-26  235.059998  235.059998  235.570007  233.330002  233.330002   
2024-11-27  234.929993  234.929993  235.690002  233.809998  234.470001   
2024-11-29  237.330002  237.330002  237.809998  233.970001  234.809998   

Price           Volume  Daily % Chang

### 5. Negating Conditions with `~`

Filter rows where Adj Close is *not* greater than 150:

In [ ]:
# Negating a condition
filtered_df = df_aapl[~(df_aapl['Adj Close'] > 150)]
print(filtered_df)

# Look at how efficient this can be!

Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2005-01-03    0.953359    1.130179    1.162679    1.117857    1.156786   
2005-01-04    0.963151    1.141786    1.169107    1.124464    1.139107   
2005-01-05    0.971586    1.151786    1.165179    1.143750    1.151071   
2005-01-06    0.972339    1.152679    1.159107    1.130893    1.154821   
2005-01-07    1.043136    1.236607    1.243393    1.156250    1.160714   
...                ...         ...         ...         ...         ...   
2023-03-02  144.612061  145.910004  146.710007  143.899994  144.380005   
2023-03-03  149.686508  151.029999  151.110001  147.330002  148.039993   
2023-03-09  149.250412  150.589996  154.539993  150.229996  153.559998   
2023-03-10  147.179001  148.500000  150.940002  147.610001  150.210007   
2023-03-13  149.131500  150.470001  153.139999  147.699997  147.809998   

Price           Volume  Daily % Chang

### 6. Keeping Only Specific Columns After Filtering
So far, we have kept all columns after filtering. But what if we wanted to filter based on a condition, yet keep only some of the columns?

Example: Keep Only Adj Close and Volume

After filtering, you can select specific columns by chaining [['col1', 'col2']]:

In [ ]:
# Filter rows and keep only Adj Close and Volume
filtered_df = df_aapl[df_aapl['Adj Close'] > 150][['Adj Close', 'Volume']]
print(filtered_df)


Price        Adj Close    Volume
Date                            
2021-08-30  150.432648  90956700
2021-09-02  150.953339  71115500
2021-09-03  151.591934  57808700
2021-09-07  153.940033  82278300
2021-09-08  152.387726  74420200
...                ...       ...
2024-11-22  229.869995  38168300
2024-11-25  232.869995  90152800
2024-11-26  235.059998  45986200
2024-11-27  234.929993  33498400
2024-11-29  237.330002  28481400

[617 rows x 2 columns]


Example: Multiple Conditions with Selected Columns

Filter rows where Adj Close > 150 and Volume > 1,000,000, and keep only
Adj Close and Daily % Change:

In [ ]:
# Filter rows and keep specific columns
filtered_df = df_aapl[(df_aapl['Adj Close'] > 150) & (df_aapl['Volume'] > 1_000_000)][['Adj Close', 'Daily % Change']]
print(filtered_df)


Price        Adj Close  Daily % Change
Date                                  
2021-08-30  150.432648        0.030417
2021-09-02  150.953339        0.007475
2021-09-03  151.591934        0.004230
2021-09-07  153.940033        0.015490
2021-09-08  152.387726       -0.010084
...                ...             ...
2024-11-22  229.869995        0.005908
2024-11-25  232.869995        0.013051
2024-11-26  235.059998        0.009404
2024-11-27  234.929993       -0.000553
2024-11-29  237.330002        0.010216

[617 rows x 2 columns]


### Summary

1. Chaining Conditions:

- Always use parentheses () around each condition.
- Combine conditions with & (AND), | (OR), and ~ (NOT).

2. Extracting Specific Columns:

- Use [['col1', 'col2']] to refine the filtered DataFrame to include only relevant columns.

3. Boolean Masks:

- Conditions generate boolean masks that filter rows efficiently.

4. Avoid Common Errors:

- Do not use Python's and/or operators. Instead, use Pandas-specific &/| for element-wise comparisons.

**In filtering, most errors occur because of bad syntax and not placing the parentheses and square brackets at the correct places.**

### Common Uses and Examples

This section will present you with some common uses of filtering with time series data.



1. Identifying days with high volume

In [ ]:
high_volume_df = df_aapl[df_aapl['Volume'] > 50_000_000]
high_volume_df

# Python Trick: you can use underscores in really large numbers as a thousand separator to make it more
# visually applealing.

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN
2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271
2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758
2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811
...,...,...,...,...,...,...,...
2024-10-23,230.506393,230.759995,235.139999,227.759995,234.080002,52287000,-0.021623
2024-10-31,225.661728,225.910004,229.830002,225.369995,229.339996,64370100,-0.018209
2024-11-01,222.665024,222.910004,225.350006,220.270004,220.970001,65276700,-0.013280


2. Filter Stocks with Significant Daily Changes:

In [ ]:
big_changes_df = df_aapl[abs(df_aapl['Daily % Change']) > 0.02]
big_changes_df

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811
2005-01-11,0.972489,1.152857,1.234821,1.145357,1.218750,2611627200,-0.063805
2005-01-13,1.051422,1.246429,1.328929,1.245179,1.316250,3164716800,0.066300
2005-01-31,1.158370,1.373214,1.390893,1.330536,1.331786,1681097600,0.039470
2005-02-02,1.199494,1.421964,1.426964,1.387321,1.391964,1020062400,0.027086
...,...,...,...,...,...,...,...
2024-09-30,232.743942,233.000000,233.000000,229.649994,230.039993,54541900,0.022872
2024-10-01,225.961411,226.210007,229.649994,223.740005,229.520004,63285000,-0.029142
2024-10-07,221.446365,221.690002,225.690002,221.330002,224.500000,39505400,-0.022531


3. Refine Analysis by Selecting Columns:

In [ ]:
specific_df = df_aapl[(df_aapl['Adj Close'] > 150) & (df_aapl['Volume'] > 1_000_000)][['Adj Close', 'Volume']]
specific_df

Price,Adj Close,Volume
Date,,
2021-08-30,150.432648,90956700
2021-09-02,150.953339,71115500
2021-09-03,151.591934,57808700
2021-09-07,153.940033,82278300
2021-09-08,152.387726,74420200
...,...,...
2024-11-22,229.869995,38168300
2024-11-25,232.869995,90152800
2024-11-26,235.059998,45986200


These are just some exmaples of how we will use filters. We will also use them in conjunction with `.loc` and `.iloc` to help filter data between specific dates.

Imagine this scenario:
You are a quantitative reasearcher, and you want to know how many times in each calendar year since 2000 that SPY had at least a -10% correction.

How would you do that? You will need to use multiple filtering methods and method chaining.

This is the kind of task that you will learn how to do in our courses.

It is time to move onto one of the most important concepts that you need to master with dataframes: Vectorization.

## Introduction to Vectorization in Pandas

Vectorization is one of the most powerful concepts in data analysis, allowing operations to be applied across entire datasets at once, rather than iterating through individual elements. This approach leverages the optimized, low-level implementation of operations in libraries like Pandas and NumPy, resulting in significant speed improvements and more concise, readable code.

**Why Vectorization Is Important**

1. Speed:

- Vectorized operations are highly optimized and can be orders of magnitude faster than equivalent operations using Python loops.
- They take advantage of compiled C code under the hood, avoiding the overhead of Python's interpreted loops.

2. Simplicity:

- Code using vectorization is more concise, easier to read, and less error-prone compared to loops.
- A single line of vectorized code can replace many lines of loop-based logic.

3. Scalability:

- Vectorized operations scale well to large datasets, which is critical in finance when dealing with millions of rows of time-series or trade data.

In financial analysis, where time and accuracy are paramount, understanding and applying vectorization ensures your code remains efficient and professional.

We will continue to work with `df_aapl` and show some common operations.

### 1. Adding, Subtracting, Multiplying, Dividing Entire Columns

Perform arithmetic operations across entire columns.

In [ ]:
# Calculate a 10% increase in Adj Close prices
df_aapl['Adj Close + 10%'] = df_aapl['Adj Close'] * 1.10

# Calculate the difference between High and Low prices
df_aapl['High - Low'] = df_aapl['High'] - df_aapl['Low']

# Convert Volume to millions
df_aapl['Volume (M)'] = df_aapl['Volume'] / 1_000_000

df_aapl.tail()


Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M)
Date,,,,,,,,,,
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300,0.005908,252.856995,2.660004,38.1683
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800,0.013051,256.156995,3.509995,90.1528
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200,0.009404,258.565997,2.240005,45.9862
2024-11-27,234.929993,234.929993,235.690002,233.809998,234.470001,33498400,-0.000553,258.422992,1.880005,33.4984
2024-11-29,237.330002,237.330002,237.809998,233.970001,234.809998,28481400,0.010216,261.063002,3.839996,28.4814


I hope you see the power of this. We were able to execute operations across all rows for specified columns in one line of code.

The alternative would be to use a for loop, going through each row and applying the operation one at a time! You must get used to vectorization if you want to have efficient and professional code.

### 2. Applying Mathematical Functions

While we haven't discussed the numpy library yet, we will use a couple of common methods for demonstration purposes.

- Use functions like np.log(), np.sqrt(), or Pandas .abs() for transformations.

In [ ]:
import numpy as np

# Logarithm of Adj Close
df_aapl['Log(Adj Close)'] = np.log(df_aapl['Adj Close'])

# Absolute values of Daily % Change
df_aapl['Abs(Daily % Change)'] = df_aapl['Daily % Change'].abs()

df_aapl.tail()

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change)
Date,,,,,,,,,,,,
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300,0.005908,252.856995,2.660004,38.1683,5.437514,0.005908
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800,0.013051,256.156995,3.509995,90.1528,5.450480,0.013051
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200,0.009404,258.565997,2.240005,45.9862,5.459841,0.009404
2024-11-27,234.929993,234.929993,235.690002,233.809998,234.470001,33498400,-0.000553,258.422992,1.880005,33.4984,5.459288,0.000553
2024-11-29,237.330002,237.330002,237.809998,233.970001,234.809998,28481400,0.010216,261.063002,3.839996,28.4814,5.469452,0.010216


Once again you see how easy it is to add a new column to our dataframe based on a mathematical function or transformation of our existing data.

You could of course create a new dataframe instead of adding new columns.

### 3. Boolean Filtering with Vectorized Conditions

This is exactly what we did before for filtering, but we are showing you how that in itself is a vectorization operation!

Look how fast these operations are performed!

In [ ]:
# Identify rows where Adj Close > 150 and Volume > 10 million
high_value_days = df_aapl[(df_aapl['Adj Close'] > 150) & (df_aapl['Volume'] > 10_000_000)]
high_value_days

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change)
Date,,,,,,,,,,,,
2021-08-30,150.432648,153.119995,153.490005,148.610001,149.000000,90956700,0.030417,165.475912,4.880005,90.9567,5.013515,0.030417
2021-09-02,150.953339,153.649994,154.720001,152.399994,153.869995,71115500,0.007475,166.048672,2.320007,71.1155,5.016971,0.007475
2021-09-03,151.591934,154.300003,154.630005,153.089996,153.759995,57808700,0.004230,166.751128,1.540009,57.8087,5.021192,0.004230
2021-09-07,153.940033,156.690002,157.259995,154.389999,154.970001,82278300,0.015490,169.334036,2.869995,82.2783,5.036563,0.015490
2021-09-08,152.387726,155.110001,157.039993,153.979996,156.979996,74420200,-0.010084,167.626498,3.059998,74.4202,5.026428,0.010084
...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300,0.005908,252.856995,2.660004,38.1683,5.437514,0.005908
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800,0.013051,256.156995,3.509995,90.1528,5.450480,0.013051
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200,0.009404,258.565997,2.240005,45.9862,5.459841,0.009404


### 4. Conditional Assignment

Again, while we haven't walked through numpy yet, here we will show you the Python equivalent of the Excel If statement:

This is the excel statement we all know and love:

`IF(condition to evaluate, output if true, output if false)`

In Python, we use np.where:

`np.where(condition to evaluate, output if true, output if false)`

In [ ]:
# Add a column marking whether Adj Close > 150
df_aapl['Above 150'] = np.where(df_aapl['Adj Close'] > 150, 'Yes', 'No')
df_aapl.tail()

# Look at the last column; You will see either Yes or No. You could then filter based on the Yes or No condition!

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150
Date,,,,,,,,,,,,,
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300,0.005908,252.856995,2.660004,38.1683,5.437514,0.005908,Yes
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800,0.013051,256.156995,3.509995,90.1528,5.450480,0.013051,Yes
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200,0.009404,258.565997,2.240005,45.9862,5.459841,0.009404,Yes
2024-11-27,234.929993,234.929993,235.690002,233.809998,234.470001,33498400,-0.000553,258.422992,1.880005,33.4984,5.459288,0.000553,Yes
2024-11-29,237.330002,237.330002,237.809998,233.970001,234.809998,28481400,0.010216,261.063002,3.839996,28.4814,5.469452,0.010216,Yes


### 5. Using `.sub`, `.add`, `.mul`, and `.div` in Pandas

Pandas provides arithmetic methods like `.sub()`, `.add()`, `.mul()`, and `.div()` as alternatives to mathematical operators (-, +, *, /).

These methods offer additional flexibility, such as handling mismatched indices and applying operations with specific alignment.

**When to Use Arithmetic Methods Instead of Operators**
1. Index Alignment:

- These methods align rows and columns based on their labels, ensuring accurate operations even with mismatched indices or column names.

2. Adding Fill Values for Missing Data:

- They allow you to specify a fill_value to handle missing data, which is not possible with basic operators.

3. Clarity:

- Using named methods can make your intentions clearer in more complex operations.

---

Basic Arithmetic Using Methods

Perform arithmetic across columns in df_aapl.

In [ ]:
# Let's create a subset of df_appl so that we don't have a messy df with too many columns
# You will also learn a best-practices technique to create a new df with a subset of columns from the original

columns_to_keep = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
df_aapl_copy = df_aapl[columns_to_keep]  # notice how columns_to_keep is a list, which is required

# Subtract 'Low' from 'High' to calculate daily range
df_aapl_copy['Range'] = df_aapl_copy['High'].sub(df_aapl_copy['Low'])

# Add 10% to 'Adj Close'
df_aapl_copy['Adj Close + 10%'] = df_aapl_copy['Adj Close'].mul(1.10)

# Divide 'Volume' by 1,000,000 to convert to millions
df_aapl_copy['Volume (M)'] = df_aapl_copy['Volume'].div(1_000_000)

# Add 'High' and 'Low' to calculate their sum
df_aapl_copy['High + Low'] = df_aapl_copy['High'].add(df_aapl_copy['Low'])

df_aapl_copy

<ipython-input-69-d0e5caefda71>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_aapl_copy['Range'] = df_aapl_copy['High'].sub(df_aapl_copy['Low'])
<ipython-input-69-d0e5caefda71>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_aapl_copy['Adj Close + 10%'] = df_aapl_copy['Adj Close'].mul(1.10)
<ipython-input-69-d0e5caefda71>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

Price,Open,High,Low,Close,Adj Close,Volume,Range,Adj Close + 10%,Volume (M),High + Low
Date,,,,,,,,,,
2005-01-03,1.156786,1.162679,1.117857,1.130179,0.953359,691992000,0.044822,1.048695,691.9920,2.280536
2005-01-04,1.139107,1.169107,1.124464,1.141786,0.963151,1096810400,0.044643,1.059466,1096.8104,2.293571
2005-01-05,1.151071,1.165179,1.143750,1.151786,0.971586,680433600,0.021429,1.068744,680.4336,2.308929
2005-01-06,1.154821,1.159107,1.130893,1.152679,0.972339,705555200,0.028214,1.069573,705.5552,2.290000
2005-01-07,1.160714,1.243393,1.156250,1.236607,1.043136,2227450400,0.087143,1.147450,2227.4504,2.399643
...,...,...,...,...,...,...,...,...,...,...
2024-11-22,228.059998,230.720001,228.059998,229.869995,229.869995,38168300,2.660004,252.856995,38.1683,458.779999
2024-11-25,231.460007,233.250000,229.740005,232.869995,232.869995,90152800,3.509995,256.156995,90.1528,462.990005
2024-11-26,233.330002,235.570007,233.330002,235.059998,235.059998,45986200,2.240005,258.565997,45.9862,468.900009


### **Important: SettingWithCopyWarning**

You likely are getting setting with copy warnings. We need to address that now so that you understand what is causing it and how to avoid it, since you will likely encounter it OFTEN when coding.

The SettingWithCopyWarning appears when you attempt to modify a slice of a DataFrame, which may be a view rather than a new, independent copy. This warning is Pandas' way of alerting you that changes might not behave as expected. For example, modifying the slice may or may not affect the original DataFrame, leading to unpredictable results.

Essentially, Pandas can get confused when doing these kinds of operations. This is why, it often better to create a copy of a dataframe using `df2 = df1.copy()` and then working with the copy.

So, always check whether you are working with a copy or a slice of the original DataFrame. Using .copy() or .loc[] ensures clarity and avoids unexpected behavior. **This is critical for maintaining data integrity**, especially when modifying filtered subsets of your data.

Here are the "best practices" to deal with this:

1. Use `.copy()`: When creating a new DataFrame from a filtered slice, always use `.copy()` to ensure you're working with an independent copy.

 - Example: `df_aapl_copy = df_aapl[df_aapl['Adj Close'] > 150].copy()`

2. Use `.loc[]` for Assignment: When modifying values in a DataFrame, use `.loc[]` to explicitly specify the rows and columns you want to update.

 - Example: `df_aapl.loc[df_aapl['Adj Close'] > 150, 'Volume (M)'] = df_aapl['Volume'] / 1_000_000`

3. Understand the Warning: If you're certain your changes won't affect the original DataFrame, the warning can usually be ignored. However, using `.copy()` is safer and avoids ambiguity.

### The Proof of the Pudding is in the Tasting
We said that vectorization is fast. And now we will prove it, and you will also learn how to time operations in Python.

Remember that we said the alternative here is using for loops? Let's try that out and time how long it takes.

We will use the example of adding 10% to all Adj Close prices in df_aapl

In [ ]:
import time

# Vectorized approach
start_vec = time.time()
df_aapl['Adj Close + 10%'] = df_aapl['Adj Close'] * 1.10
end_vec = time.time()

# Print timing results
print("Vectorized approach time:", end_vec - start_vec)

Vectorized approach time: 0.001802682876586914


In [ ]:
# Loop-based approach
start_loop = time.time()
adj_close_10pct = []
for price in df_aapl['Adj Close']:
    adj_close_10pct.append(price * 1.10)
df_aapl['Adj Close + 10% (Loop)'] = adj_close_10pct
end_loop = time.time()

print("Loop-based approach time:", end_loop - start_loop)

# Comparing speeds: Vectorized vs Loop
percent_diff = (end_loop - start_loop) / (end_vec - start_vec)
print(f"The vectorized approach is {np.round(percent_diff, 2)} times faster than the loop!")


Loop-based approach time: 0.010066986083984375
The vectorized approach is 5.58 times faster than the loop!


## Index Sorting and Operations in Pandas

Efficiently managing the index of a DataFrame and sorting data are essential for organizing and analyzing large datasets. The methods `.reset_index()`, `.set_index()`, `.sort_index()`, and sorting by a column allow you to structure and retrieve data in meaningful ways.


### 1. `reset_index()`

**What It Does**

reset_index() moves the current index (e.g., DatetimeIndex) to a regular column and resets the index to the default integer-based one.

In [ ]:
# Reset index to default integers and keep the original index as a column
df_reset = df_aapl.reset_index()
df_reset.head()


Price,Date,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150,Adj Close + 10% (Loop)
0,2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN,1.048695,0.044822,691.9920,-0.047764,NaN,No,1.048695
1,2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271,1.059466,0.044643,1096.8104,-0.037545,0.010271,No,1.059466
2,2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758,1.068744,0.021429,680.4336,-0.028826,0.008758,No,1.068744
3,2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775,1.069573,0.028214,705.5552,-0.028051,0.000775,No,1.069573
4,2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811,1.147450,0.087143,2227.4504,0.042232,0.072811,No,1.147450


You can see that the datetime index 'Date' is now a column in the dataframe and the index is now ordered integers.

There is also the option to drop the 'Date' column so that we reset the index to integers and then remove the Date column from the resulting dataframe:

In [ ]:
df_reset = df_aapl.reset_index(drop=True)  # note that drop=False is the default
df_reset.head()

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150,Adj Close + 10% (Loop)
0,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN,1.048695,0.044822,691.9920,-0.047764,NaN,No,1.048695
1,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271,1.059466,0.044643,1096.8104,-0.037545,0.010271,No,1.059466
2,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758,1.068744,0.021429,680.4336,-0.028826,0.008758,No,1.068744
3,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775,1.069573,0.028214,705.5552,-0.028051,0.000775,No,1.069573
4,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811,1.147450,0.087143,2227.4504,0.042232,0.072811,No,1.147450


**Why It’s Important**

- Flattening the DataFrame: Makes the DataFrame easier to work with when the index isn’t needed for operations like merging or exporting.
- Flexibility: Enables you to manipulate or reassign the index without losing information.

### 2. `set_index()`

**What It Does**

`set_index()` assigns one or more columns as the new index of the DataFrame.



In [ ]:
# Set the 'Date' column as the new index
df_set = df_aapl.reset_index().set_index('Date')
df_set.head()


Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150,Adj Close + 10% (Loop)
Date,,,,,,,,,,,,,,
2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN,1.048695,0.044822,691.9920,-0.047764,NaN,No,1.048695
2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271,1.059466,0.044643,1096.8104,-0.037545,0.010271,No,1.059466
2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758,1.068744,0.021429,680.4336,-0.028826,0.008758,No,1.068744
2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775,1.069573,0.028214,705.5552,-0.028051,0.000775,No,1.069573
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811,1.147450,0.087143,2227.4504,0.042232,0.072811,No,1.147450


ok, the above example might seem a little odd. We reset the index and made 'Date' a column, and then we assigned the 'Date' column as the index, thus restoring the orignial structure.

Notice that we also created a new dataframe by using `df_set =`

But what if we just wanted to work with the original one and not create a new one.

We could have done this:

In [ ]:
df_aapl = df_aapl.reset_index()
df_aapl = df_aapl.set_index('Date')
df_aapl

# We are essentially creating a copy of df_aapl and calling it df_aapl???
# That's not efficient.

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150,Adj Close + 10% (Loop)
Date,,,,,,,,,,,,,,
2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN,1.048695,0.044822,691.9920,-0.047764,NaN,No,1.048695
2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271,1.059466,0.044643,1096.8104,-0.037545,0.010271,No,1.059466
2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758,1.068744,0.021429,680.4336,-0.028826,0.008758,No,1.068744
2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775,1.069573,0.028214,705.5552,-0.028051,0.000775,No,1.069573
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811,1.147450,0.087143,2227.4504,0.042232,0.072811,No,1.147450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300,0.005908,252.856995,2.660004,38.1683,5.437514,0.005908,Yes,252.856995
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800,0.013051,256.156995,3.509995,90.1528,5.450480,0.013051,Yes,256.156995
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200,0.009404,258.565997,2.240005,45.9862,5.459841,0.009404,Yes,258.565997


But that could also be done using `inplace=True` so that we don't create a copy and work directly on the original.

In [ ]:
df_aapl.reset_index(drop=False, inplace=True)
df_aapl.head()

# In the output you will see we accomplished the exact smae thing

Price,Date,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150,Adj Close + 10% (Loop)
0,2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN,1.048695,0.044822,691.9920,-0.047764,NaN,No,1.048695
1,2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271,1.059466,0.044643,1096.8104,-0.037545,0.010271,No,1.059466
2,2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758,1.068744,0.021429,680.4336,-0.028826,0.008758,No,1.068744
3,2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775,1.069573,0.028214,705.5552,-0.028051,0.000775,No,1.069573
4,2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811,1.147450,0.087143,2227.4504,0.042232,0.072811,No,1.147450


And now to set the index using inplace:

In [ ]:
df_aapl.set_index('Date', inplace=True)
df_aapl.head()

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150,Adj Close + 10% (Loop)
Date,,,,,,,,,,,,,,
2005-01-03,0.953359,1.130179,1.162679,1.117857,1.156786,691992000,NaN,1.048695,0.044822,691.9920,-0.047764,NaN,No,1.048695
2005-01-04,0.963151,1.141786,1.169107,1.124464,1.139107,1096810400,0.010271,1.059466,0.044643,1096.8104,-0.037545,0.010271,No,1.059466
2005-01-05,0.971586,1.151786,1.165179,1.143750,1.151071,680433600,0.008758,1.068744,0.021429,680.4336,-0.028826,0.008758,No,1.068744
2005-01-06,0.972339,1.152679,1.159107,1.130893,1.154821,705555200,0.000775,1.069573,0.028214,705.5552,-0.028051,0.000775,No,1.069573
2005-01-07,1.043136,1.236607,1.243393,1.156250,1.160714,2227450400,0.072811,1.147450,0.087143,2227.4504,0.042232,0.072811,No,1.147450


### 3. `sort_index()`

**What It Does**

`sort_index()` reorders the rows (or columns) in ascending or descending order based on the index.

In [ ]:
# Sort the index in ascending order
df_sorted_index = df_aapl.sort_index()

# Sort the index in descending order
df_sorted_desc = df_aapl.sort_index(ascending=False)

df_sorted_desc.head()

# Notice the dataframe is now ordered by date in descending order

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150,Adj Close + 10% (Loop)
Date,,,,,,,,,,,,,,
2024-11-29,237.330002,237.330002,237.809998,233.970001,234.809998,28481400,0.010216,261.063002,3.839996,28.4814,5.469452,0.010216,Yes,261.063002
2024-11-27,234.929993,234.929993,235.690002,233.809998,234.470001,33498400,-0.000553,258.422992,1.880005,33.4984,5.459288,0.000553,Yes,258.422992
2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200,0.009404,258.565997,2.240005,45.9862,5.459841,0.009404,Yes,258.565997
2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800,0.013051,256.156995,3.509995,90.1528,5.450480,0.013051,Yes,256.156995
2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300,0.005908,252.856995,2.660004,38.1683,5.437514,0.005908,Yes,252.856995


**Key Options:**

- axis=0: Sort rows (default).
- axis=1: Sort columns.
- inplace=True: Modify the original DataFrame.

**Why It’s Important**
- Consistency: Ensures the DataFrame is ordered correctly by its index, critical for time-series data.
- Readability: Makes the data easier to interpret when the index is in order.

### 4. Sorting by a Column

**What It Does**

Sort rows based on the values in one or more columns.

In [ ]:
# Sort by 'Adj Close' in ascending order
df_sorted = df_aapl.sort_values(by='Adj Close')
print(df_sorted.head())

Price       Adj Close     Close      High       Low      Open      Volume  \
Date                                                                        
2005-01-03   0.953359  1.130179  1.162679  1.117857  1.156786   691992000   
2005-01-04   0.963151  1.141786  1.169107  1.124464  1.139107  1096810400   
2005-01-05   0.971586  1.151786  1.165179  1.143750  1.151071   680433600   
2005-01-06   0.972339  1.152679  1.159107  1.130893  1.154821   705555200   
2005-01-11   0.972489  1.152857  1.234821  1.145357  1.218750  2611627200   

Price       Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
Date                                                                  
2005-01-03             NaN         1.048695    0.044822    691.9920   
2005-01-04        0.010271         1.059466    0.044643   1096.8104   
2005-01-05        0.008758         1.068744    0.021429    680.4336   
2005-01-06        0.000775         1.069573    0.028214    705.5552   
2005-01-11       -0.063805        

In [ ]:
# Sort by 'Adj Close' in descending order
df_sorted_desc = df_aapl.sort_values(by='Adj Close', ascending=False)
print(df_sorted_desc.head())


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2024-11-29  237.330002  237.330002  237.809998  233.970001  234.809998   
2024-10-21  236.220108  236.479996  236.850006  234.449997  234.449997   
2024-10-22  235.600800  235.860001  236.220001  232.600006  233.889999   
2024-11-26  235.059998  235.059998  235.570007  233.330002  233.330002   
2024-11-27  234.929993  234.929993  235.690002  233.809998  234.470001   

Price         Volume  Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
Date                                                                            
2024-11-29  28481400        0.010216       261.063002    3.839996     28.4814   
2024-10-21  36254500        0.006298       259.842119    2.400009     36.2545   
2024-10-22  38846600       -0.002622       259.160880    3.619995     38.8466   
2024-11-26  45986200        0.009404       258.565997    2.240005     45.986

In [ ]:
# Sort by multiple columns: 'Volume' (descending), then 'Adj Close' (ascending)
df_sorted_multi = df_aapl.sort_values(by=['Volume', 'Adj Close'], ascending=[False, True])
print(df_sorted_multi.head())

Price       Adj Close     Close      High       Low      Open      Volume  \
Date                                                                        
2008-01-23   4.189718  4.966786  5.000000  4.505000  4.863929  3372969600   
2007-01-09   2.788827  3.306071  3.320714  3.041071  3.087500  3349298400   
2005-01-13   1.051422  1.246429  1.328929  1.245179  1.316250  3164716800   
2007-01-10   2.922289  3.464286  3.492857  3.337500  3.383929  2952880000   
2005-04-14   1.122520  1.330714  1.412857  1.315714  1.386071  2753192400   

Price       Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
Date                                                                  
2008-01-23       -0.106464         4.608690    0.495000   3372.9696   
2007-01-09        0.083070         3.067709    0.279643   3349.2984   
2005-01-13        0.066300         1.156564    0.083750   3164.7168   
2007-01-10        0.047856         3.214518    0.155357   2952.8800   
2005-04-14       -0.092106        

**Why It’s Important**
- Prioritization: Helps identify rows based on ranking, such as the highest volume trading days or the lowest closing prices.
- Preparation: Essential for tasks like ranking, filtering top/bottom rows, or visualizing sorted data.

## Dealing with NaN Values in Pandas

`NaN` values represent missing or undefined data, which is common in financial datasets due to market holidays, incomplete data feeds, or errors. Handling these values correctly is essential to maintain the accuracy and reliability of your analysis.

### 1. Finding `NaN` Values

**What It Does**

Identifies where missing values (NaN) exist in a DataFrame or Series.

**How to Use It**

i. Check for Any Missing Values


In [ ]:
print(df_aapl.isnull())

# Returns a DataFrame of the same shape, where True indicates missing values.
# This isn't exactly helpful...

Price       Adj Close  Close   High    Low   Open  Volume  Daily % Change  \
Date                                                                        
2005-01-03      False  False  False  False  False   False            True   
2005-01-04      False  False  False  False  False   False           False   
2005-01-05      False  False  False  False  False   False           False   
2005-01-06      False  False  False  False  False   False           False   
2005-01-07      False  False  False  False  False   False           False   
...               ...    ...    ...    ...    ...     ...             ...   
2024-11-22      False  False  False  False  False   False           False   
2024-11-25      False  False  False  False  False   False           False   
2024-11-26      False  False  False  False  False   False           False   
2024-11-27      False  False  False  False  False   False           False   
2024-11-29      False  False  False  False  False   False           False   

ii. Check if any missing values exist

In [ ]:
print(df_aapl.isnull().any())
# This is a much more useful output!

Price
Adj Close                 False
Close                     False
High                      False
Low                       False
Open                      False
Volume                    False
Daily % Change             True
Adj Close + 10%           False
High - Low                False
Volume (M)                False
Log(Adj Close)            False
Abs(Daily % Change)        True
Above 150                 False
Adj Close + 10% (Loop)    False
dtype: bool


iii. Count missing values per column

In [ ]:
print(df_aapl.isnull().sum())


Price
Adj Close                 0
Close                     0
High                      0
Low                       0
Open                      0
Volume                    0
Daily % Change            1
Adj Close + 10%           0
High - Low                0
Volume (M)                0
Log(Adj Close)            0
Abs(Daily % Change)       1
Above 150                 0
Adj Close + 10% (Loop)    0
dtype: int64


This output makes perfect sense. We only have to missing values. This occured when we did the percent change calculations, and the first row gives an NaN since there were no rows above on which to calculate the percent change.

**Why It’s Important**

Locates gaps in data, helping you decide how to address missing values.
Ensures that methods like mean() or pct_change() are not skewed by NaN values.

So now that we know how to detect NaN, what do we do about them?

Before we continue, let's create a sample dataframe with random data. This will make all subsequent examples clearer.


In [ ]:
# Create a date range for the index
date_range = pd.date_range(start="2023-01-01", periods=100, freq="D")

# Generate random numeric data with NaN values
np.random.seed(42)
data = {
    "Price": np.random.choice([np.nan, 100, 105, 110], size=100),
    "Volume": np.random.choice([np.nan, 1000, 1200, 1500], size=100),
    "Returns": np.random.choice([np.nan, 0.01, -0.02, 0.03], size=100)
}

# Create a DataFrame
sample_df = pd.DataFrame(data, index=date_range)

sample_df.head()

,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,NaN,1000.0,-0.02
2023-01-04,105.0,1500.0,NaN
2023-01-05,105.0,1000.0,0.03


Feel free to add some cells here and explore the sample data. We will use `.info()` to show that there are missing values

In [ ]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    80 non-null     float64
 1   Volume   74 non-null     float64
 2   Returns  72 non-null     float64
dtypes: float64(3)
memory usage: 3.1 KB


### 2. `dropna()`

**What It Does**
- Removes rows or columns that contain missing values.

**i. Drop rows that have ANY missing values**

In [ ]:
# This drops all rows that have at least one missing value
# we will create a new df for these examples and not use inplace=True

# 1) Default, axis=0 --> drop rows that have at least one NaN
df_dropped_axis0 = sample_df.dropna()
df_dropped_axis0.info()
df_dropped_axis0.head(15)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 40 entries, 2023-01-01 to 2023-04-08
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    40 non-null     float64
 1   Volume   40 non-null     float64
 2   Returns  40 non-null     float64
dtypes: float64(3)
memory usage: 1.2 KB


,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-05,105.0,1000.0,0.03
2023-01-09,105.0,1000.0,0.01
2023-01-11,105.0,1500.0,-0.02
2023-01-12,105.0,1200.0,-0.02
2023-01-14,105.0,1000.0,-0.02
2023-01-15,110.0,1200.0,-0.02
2023-01-19,110.0,1500.0,0.03
2023-01-21,100.0,1500.0,-0.02


We can see that there are many missing days now as they have been removed.

The `.info()` shows that we now have only 40 days, and that 60 were dropped!

In [ ]:
# This drops all rows that have at least one missing value
# we will create a new df for these examples and not use inplace=True

# 2) axis=1 --> drop columns that have at least one NaN
df_dropped_axis1 = sample_df.dropna(axis=1)
df_dropped_axis1.info()
df_dropped_axis1.head(15)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Empty DataFrame


""
2023-01-01
2023-01-02
2023-01-03
2023-01-04
2023-01-05
2023-01-06
2023-01-07
2023-01-08
2023-01-09
2023-01-10


We got an empty dataframe! This should come as no surprise because all columns had at least one NaN

**ii. Dropping rows only if ALL values are NaN**

What if we didn't want to drop all rows if they had only one NaN? This isn't what we always want to do.

The next example shows how you can instruct Python to delete rows that have ALL NaNs in a row/column. This means that every value in the row must be NaN f to be removed.

In this case, we add the parameter 'how': `dropna(how='all')`

In [ ]:
df_drop_all = sample_df.dropna(how='all')
df_drop_all.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    80 non-null     float64
 1   Volume   74 non-null     float64
 2   Returns  72 non-null     float64
dtypes: float64(3)
memory usage: 3.1 KB


In this case, no rows were dropped since there were no rows that contained NaN for each column/value.
We would expect the same result for:
`dropna(axis=1, how='all')`

Note: the default behaviour is: `dropna(how='any')` which is the same as `dropna()`

**iii. Dropping rows/columns only if there are a minimum amount of missing values**

In [ ]:
# This will remove rows that have at least 2 NaN values
df_thresh = sample_df.dropna(thresh=2)
df_thresh.info()

# Compare against the original
sample_df.info()
# You will see that we removed a total of 14 rows

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 86 entries, 2023-01-01 to 2023-04-10
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    75 non-null     float64
 1   Volume   69 non-null     float64
 2   Returns  68 non-null     float64
dtypes: float64(3)
memory usage: 2.7 KB
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    80 non-null     float64
 1   Volume   74 non-null     float64
 2   Returns  72 non-null     float64
dtypes: float64(3)
memory usage: 3.1 KB


### 3. `fillna()`

We might not always want to delete the rows/columns with NaN values.
Very often, we need to do something with them so that we don't lose whole rows of valuable data.

We will look at the most popular ways that you can deal with missing values.

**i. Fill with a fixed value (example: fill all NaN with 0)**

In [ ]:
df_filled_0 = sample_df.fillna(0)
df_filled_0.info()
df_filled_0.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    100 non-null    float64
 1   Volume   100 non-null    float64
 2   Returns  100 non-null    float64
dtypes: float64(3)
memory usage: 3.1 KB


,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,0.0,1000.0,-0.02
2023-01-04,105.0,1500.0,0.00
2023-01-05,105.0,1000.0,0.03


**ii. Forward fill with `ffill`**

A forward fill will take the most recent valid value and use that.
Look at the original, and then predict what the result will be once we do a forward fill:

In [ ]:
sample_df.head()

,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,NaN,1000.0,-0.02
2023-01-04,105.0,1500.0,NaN
2023-01-05,105.0,1000.0,0.03


In [ ]:
df_ffill = sample_df.fillna(method='ffill')
df_ffill.head()

<ipython-input-93-063c1db7019e>:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_ffill = sample_df.fillna(method='ffill')


,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,110.0,1000.0,-0.02
2023-01-04,105.0,1500.0,-0.02
2023-01-05,105.0,1000.0,0.03


**iii. Backward fill with `bfill`**


In [ ]:
df_bfill = sample_df.fillna(method='bfill')
df_bfill.head()

<ipython-input-94-82ca40ac0774>:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_bfill = sample_df.fillna(method='bfill')


,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,105.0,1000.0,-0.02
2023-01-04,105.0,1500.0,0.03
2023-01-05,105.0,1000.0,0.03


**iv. Filling NaN with mean or median values**

This is a very common method: fill all missing NaN with the mean values found in the column. Since we have a central tendency theorem, it makes sense to assign the mean to the missing values. This way we don;t change the overall distribution or descriptive statistics.

Of course some data sets could be skewed, and it would be more appropriate to use the median.




In [ ]:
# Let's examine the mean values first
print(sample_df.mean())

df_fill_mean = sample_df.fillna(sample_df.mean())
df_fill_mean.head()

Price       105.250000
Volume     1243.243243
Returns       0.007917
dtype: float64


,Price,Volume,Returns
2023-01-01,105.00,1200.0,-0.020000
2023-01-02,110.00,1000.0,0.030000
2023-01-03,105.25,1000.0,-0.020000
2023-01-04,105.00,1500.0,0.007917
2023-01-05,105.00,1000.0,0.030000


Add some cells and try to do this with median!

With time series, you would really have to think about the consequences
of using mean/median.

Let's take a specific situation:

Imagine you download daily pricing data for an ETF. You notice that some days have missing values for price. This could happen if there were no trades in  a given day so the data source, when detecting zero volume, might just assign an NaN to the price for that day.

What do you think is the most logical way to deal with this?

A forward fill would make sense where you just take the previous closing value and assign it to the NaN.


---



## Combining and Reshaping DataFrames

This section introduces essential methods for combining and reshaping DataFrames in Pandas. These are critical for organizing and analyzing financial datasets, especially when merging, aligning, or restructuring data from multiple sources.

We cannot stress enough how important this section is! WHile it might seem daunting at first, these are techniques which you must master!

We will explain them in very basic terms, but feel free to look at the pandas documentation or supplement explanations using a generative AI tool.

### 1. `pd.concat`

**What It Does**
- pd.concat combines multiple DataFrames along rows (axis=0) or columns (axis=1).

**When to Use It**
- Use when you want to stack DataFrames vertically (row-wise) or combine them horizontally (column-wise) without explicitly matching on an index or key.

**Parameters**
- objs: A list of DataFrames to concatenate.
- axis: Specifies the direction of concatenation:
 - 0 for rows (default).
 - 1 for columns.
- ignore_index: Resets the index in the concatenated DataFrame if True.
- keys: Adds hierarchical keys to identify the origin of each DataFrame.

pd.concat is the simplest way to combine different dataframes. But that simplicity comes at a cost. Pandas might not always know what you are trying to do. If you want to stack dataframes vertically, meaning add one dataframe to the bottom of the other, the column labels must be the exact same! Otherwise, you will see new columns added and there will be NaN.

If you want to add the second dataframe to the right of the first dataframe, the indexes must be exactly the same.

Let's start with a simple example and then we will demonstrate what happens when the labels aren't the same.

In [ ]:
# Create sample DataFrames
df1 = pd.DataFrame({"Price": [100, 105], "Volume": [1000, 1200]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"Price": [110, 115], "Volume": [1500, 1600]}, index=["2023-01-03", "2023-01-04"])

# Concatenate along rows
df_combined = pd.concat([df1, df2])  # The order of the dataframes matter! in this case, you are adding df2 below df1

# Concatenate along columns
df_combined_columns = pd.concat([df1, df2], axis=1)  # The order of the dataframes matter! in this case, you are adding df2 to the right of df1

# Concatenate with hierarchical keys
df_with_keys = pd.concat([df1, df2], keys=["DF1", "DF2"])


In [ ]:
# Let's examine the first one, stacked vertically
df_combined
# You see that the rows of df2 are now below df1

,Price,Volume
2023-01-01,100,1000
2023-01-02,105,1200
2023-01-03,110,1500
2023-01-04,115,1600


In [ ]:
# Let's examine the second one, stacked horizontally
df_combined_columns
# You see that the columns of df2 are to the right of df1.
# also notice that we have columns with the same names!

,Price,Volume,Price,Volume
2023-01-01,100.0,1000.0,NaN,NaN
2023-01-02,105.0,1200.0,NaN,NaN
2023-01-03,NaN,NaN,110.0,1500.0
2023-01-04,NaN,NaN,115.0,1600.0


In [ ]:
# We can assign a second level to the indexes with the keys parameter.
# But we will not do this very often, since it makes things complicated.
df_with_keys

Price  Volume
DF1 2023-01-01    100    1000
    2023-01-02    105    1200
DF2 2023-01-03    110    1500
    2023-01-04    115    1600

In [ ]:
# Now let's address handling the case of axis=1 where the column names are the same

# Create sample DataFrames with duplicate column names
df1 = pd.DataFrame({"Price": [100, 105], "Volume": [1000, 1200]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"Price": [110, 115], "Volume": [1500, 1600]}, index=["2023-01-01", "2023-01-02"])

# Concatenate along columns with duplicate column names
df_combined = pd.concat([df1, df2], axis=1)
print(df_combined)


            Price  Volume  Price  Volume
2023-01-01    100    1000    110    1500
2023-01-02    105    1200    115    1600


Look very closely at the difference here to the first time we did this. Now the indexes are the same in both dfs, so we don't have any NaN. Which is great!

But we don't want duplicated names. So we can handle this in two ways.
1. Add a multiindex (add a level above price and volume that identifies which df it is).

In [ ]:
df_combined = pd.concat([df1, df2], axis=1, keys=["DF1", "DF2"])
print(df_combined)


             DF1          DF2       
           Price Volume Price Volume
2023-01-01   100   1000   110   1500
2023-01-02   105   1200   115   1600


But this can complicate things later when we start doing `.loc` and `.iloc`

2. Rename the columns before or after concatenation

In [ ]:
col_names_after = ['df1_price', 'df1_volume', 'df2_price', 'df2_volume']
df_combined.columns = col_names_after
df_combined

,df1_price,df1_volume,df2_price,df2_volume
2023-01-01,100,1000,110,1500
2023-01-02,105,1200,115,1600


Let's look at what can go wrong with pd.concat when you have mismatched indexes and labels.

First, think about the context. Imagine we have two pricing dataframes, and we want to create one dataframe that has just the Adj Close columns for each. This means that we want to stack them horizontally, so we can see the Adj Close of both tickers next to each other for the same dates.

But what if you have indexes (dates) that aren't the same? Imagine one is in Canada and the other is US and they have different trading holidays? Or if one index has dates as strings anf the other index is in datetime? These kinds of problems happen all the time.


In [ ]:
# Create two DataFrames with mismatched indexes
df1 = pd.DataFrame({"PEP Price": [100, 105]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"WMT Price": [150, 160]}, index=["2023-01-03", "2023-01-04"])

# Concatenate along columns
df_combined = pd.concat([df1, df2], axis=1)
print(df_combined)


            PEP Price  WMT Price
2023-01-01      100.0        NaN
2023-01-02      105.0        NaN
2023-01-03        NaN      150.0
2023-01-04        NaN      160.0


You can see the NaN because those values don't exist. This produced a df with 4 rows when all we wanted was 2.

Now let's look at mismatched columns when the intention is to stack them vertically.

In [ ]:
# Create two DataFrames with mismatched column names
df1 = pd.DataFrame({"Price": [100, 105]})
df2 = pd.DataFrame({"Cost": [1500, 1600]})

# Concatenate along rows
df_combined = pd.concat([df1, df2], axis=0)
print(df_combined)


   Price    Cost
0  100.0     NaN
1  105.0     NaN
0    NaN  1500.0
1    NaN  1600.0


You could see that because the columns have different names, our output has 2 columns when we only wanted one.

In [ ]:
# This is what we wanted
df1 = pd.DataFrame({"Price": [100, 105]})
df2 = pd.DataFrame({"Price": [1500, 1600]})

# Concatenate along rows
df_combined = pd.concat([df1, df2], axis=0)
print(df_combined)


   Price
0    100
1    105
0   1500
1   1600




---


### 2. `df.join`

Now it is going to get interesting!

The df.join() method in Pandas is used to combine two DataFrames based on their index values. This is particularly useful when you have related datasets (e.g., stock prices and trading volumes) that share the same index (such as dates) and you want to merge them into a single DataFrame.

**What Does df.join() Do?**
- Joins two or more DataFrames together by aligning their rows based on the index.
- Adds the columns of one DataFrame to another, creating a single DataFrame.
- Allows for flexibility in handling rows that do not have matching indices (via how).

**Parameters of df.join()**
1. other:

- The DataFrame to join with the current DataFrame.

2. how:

- Determines how rows are matched between the two DataFrames:
 - 'inner': Keeps only rows with indices that are common to both DataFrames.
 - 'outer': Keeps all rows from both DataFrames, filling missing values with NaN.
 - 'left': Keeps all rows from the calling DataFrame, filling with NaN for missing rows from the other DataFrame.
 - 'right': Keeps all rows from the other DataFrame, filling with NaN for missing rows from the calling DataFrame.

3. on (Optional):

- Specifies the column(s) to join on if your DataFrame does not have an index to align on. (Typically not used with beginners.)

4. lsuffix and rsuffix:

- Add suffixes to overlapping column names from the left (calling DataFrame) and right (other DataFrame) to avoid conflicts.

Let's start with some examples. We know this seems confusing, but practice and understanding when to use it will help.

Example 1: Basic join with macthing indexes

In [ ]:
# Example 1: Basic join with matching indexes
# Create sample DataFrames with the same index
df1 = pd.DataFrame({"Price": [100, 105]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"Volume": [1000, 1200]}, index=["2023-01-01", "2023-01-02"])

# Join the two DataFrames
df_combined = df1.join(df2)
print(df_combined)


            Price  Volume
2023-01-01    100    1000
2023-01-02    105    1200


That was nice an easy! And the output is similar to pd.concat.

Example 2: Handling mismatched indexes

When the indices of the DataFrames do not perfectly align, use the how parameter to control what happens.

Inner Join

Keeps only rows where the indices match in both DataFrames.

In [ ]:
# Example 2: Mismatched indexes, inner join
df1 = pd.DataFrame({"Price": [100, 105]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"Volume": [1000, 1200]}, index=["2023-01-01", "2023-01-03"])

df_inner = df1.join(df2, how="inner")
print(df_inner)


            Price  Volume
2023-01-01    100    1000


Notice that only one index was in both dataframes, so we keep only that row in the new dataframe.

Outer Join

Keeps all rows from both DataFrames and fills missing values with NaN.

In [ ]:
# Example 3: Outer join
df_outer = df1.join(df2, how="outer")
print(df_outer)


            Price  Volume
2023-01-01  100.0  1000.0
2023-01-02  105.0     NaN
2023-01-03    NaN  1200.0


This kept all rows and fills the values in all columns with NaN if it was not present in the original dataframes.

Left Join

Keeps all rows from the calling DataFrame (df1), filling missing rows from df2 with NaN.

In [ ]:
# Example 4: Left join
# Reprint the originals so you can see what's in each
print(df1)
print(df2)

df_left = df1.join(df2, how="left")
print(df_left)


            Price
2023-01-01    100
2023-01-02    105
            Volume
2023-01-01    1000
2023-01-03    1200
            Price  Volume
2023-01-01    100  1000.0
2023-01-02    105     NaN


This kept all of the indexes in the left dataframe (df1, remember, the order matters!) and then filled the values from df2 with NaN if that index was not in df2.
2023-01-03 is not in df1, so it won't be in df_left.
2023-01-02 is not in df2, so df_left will put an NaN there.

Right Join

Keeps all rows from the other DataFrame (df2), filling missing rows from df1 with NaN.

In [ ]:
# Example 5: Right join
# Reprint the originals so you can see what's in each
print(df1)
print(df2)

df_right = df1.join(df2, how="right")
print('\ndf_right')
print(df_right)

# Compare to df_left
print('\ndf_left')
print(df_left)


            Price
2023-01-01    100
2023-01-02    105
            Volume
2023-01-01    1000
2023-01-03    1200

df_right
            Price  Volume
2023-01-01  100.0    1000
2023-01-03    NaN    1200

df_left
            Price  Volume
2023-01-01    100  1000.0
2023-01-02    105     NaN


Example 6: Handling Overlapping Column Names

When both DataFrames have columns with the same names, use the lsuffix and rsuffix parameters to differentiate them.

In [ ]:
df1 = pd.DataFrame({"Price": [100, 105]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"Price": [110, 115]}, index=["2023-01-01", "2023-01-02"])

# Add suffixes to overlapping column names
df_with_suffixes = df1.join(df2, lsuffix="_df1", rsuffix="_df2")
print(df_with_suffixes)


            Price_df1  Price_df2
2023-01-01        100        110
2023-01-02        105        115


**Why df.join() Is Important**

1. Combining Related Datasets:

- Join price data, trading volume, and returns into a single DataFrame for unified analysis.

2. Flexible Handling of Missing Data:

- The how parameter ensures you can keep all rows or only rows with matching indices based on the analysis requirements.

3. Efficient Merging on Index:

- Works directly with indices, making it faster and simpler for time-series data compared to merge() or concat().


---



### 3. `df.pivot`

The `df.pivot()` method is used to reshape a DataFrame from long format to wide format, turning unique values in a column into new column headers. This is particularly useful when working with time-series data or categorized data, such as stock prices for multiple companies.

**What Are Long and Wide Formats?**

Long and wide formats are two ways of organizing data in a DataFrame. Understanding these formats is crucial for reshaping datasets to suit specific analysis or visualization needs.

1. Long Format

In long format, the data is organized so that:

- Each row represents a single observation.
- There are multiple rows for each category or group.
- One column contains the categories or groups (e.g., stock tickers).
- Another column contains the associated values (e.g., stock prices or volumes).




In [ ]:
# Sample data in long format
data_long = {
    "Date": ["2023-01-01", "2023-01-01", "2023-01-02", "2023-01-02"],
    "Stock": ["AAPL", "MSFT", "AAPL", "MSFT"],
    "Price": [150, 250, 155, 255]
}
df_long = pd.DataFrame(data_long)
df_long

,Date,Stock,Price
0,2023-01-01,AAPL,150
1,2023-01-01,MSFT,250
2,2023-01-02,AAPL,155
3,2023-01-02,MSFT,255


Notice that you have the same dates and that all prices are stacked vertically instead of having one column for each ticker.

2. Wide Format

In wide format, the data is spread out so that:

- Each row represents a single observation or group (e.g., a date).
- Columns represent the categories or groups (e.g., stock tickers like AAPL, MSFT).
- Values are organized across multiple columns.

In [ ]:
# Convert long to wide format using pivot; this is an example; we will explain!
df_wide = df_long.pivot(index="Date", columns="Stock", values="Price")
df_wide

Stock,AAPL,MSFT
Date,,
2023-01-01,150,250
2023-01-02,155,255


Now you can see each ticker has its own column!

Let's make sure you understand when you would use each format:

1. Long Format
- Preparing data for grouping, statistical analysis, or visualization libraries like seaborn.
- Flexible for pivoting into wide format later.

2. Wide Format
- Comparing values across categories (e.g., stock prices for multiple tickers).
- Plotting time-series data in pandas or other visualization tools.

In short, wide format is what we would normally use as it is much easier to see the data. However, there are many types of graphs that we can produce only if the data is in long format.

Similarly, many machine learning algorithms cannot accept data in wide form.

Understandnig the difference between the two, and how to transofrm your data in both directions is vital!

Let's get back to examining how `.pivot()` does this for us.

**What Does df.pivot() Do?**

- Reshapes a DataFrame by reorganizing rows and columns.
- Turns a column's unique values into new column headers.
- Groups data logically for easier analysis (e.g., group stock prices by date and ticker)
- In short, `.pivot()` converts a long form dataframe to wide form.

**Parameters of df.pivot()**
1. index:

- Specifies the column(s) to use as the new row index.
- Example: Use a Date column for indexing rows.

2. columns:

- Specifies the column whose unique values will become the new column headers.
- Example: Use a Stock column to create headers for tickers like AAPL, MSFT.

3. values:

- Specifies the column whose values will populate the DataFrame's cells.
- Example: Use a Price column for stock prices in the new table.



In [ ]:
# This is the same example we showed above, converting long to wide.
# It should make more sense now!
# Create a sample long-format DataFrame
df_long = pd.DataFrame({
    "Date": ["2023-01-01", "2023-01-01", "2023-01-02", "2023-01-02"],
    "Stock": ["AAPL", "MSFT", "AAPL", "MSFT"],
    "Price": [150, 250, 155, 255]
})

# Pivot the DataFrame
df_pivoted = df_long.pivot(index="Date", columns="Stock", values="Price")
print(df_pivoted)


Stock       AAPL  MSFT
Date                  
2023-01-01   150   250
2023-01-02   155   255


Explanation:

- The Date column becomes the row index.
- The Stock column's unique values (AAPL, MSFT) become new column headers.
- The Price column populates the table with stock prices.

What if there are missing values?

If for example MSFT is missing a price on one of those dates in the Date column, .pivot will add NaN

In [ ]:
# Create a sample DataFrame with missing data
df_long = pd.DataFrame({
    "Date": ["2023-01-01", "2023-01-02", "2023-01-02"],
    "Stock": ["AAPL", "AAPL", "MSFT"],
    "Price": [150, 155, 255]
})

# Pivot the DataFrame
df_pivoted = df_long.pivot(index="Date", columns="Stock", values="Price")
print(df_pivoted)


Stock        AAPL   MSFT
Date                    
2023-01-01  150.0    NaN
2023-01-02  155.0  255.0


**Common Error with df.pivot()**
This is one of the most common errors that you will encounter:

Duplicate Entries in Index-Column Pairs

- If there are duplicate combinations of index and columns, df.pivot() will raise a ValueError.

In [ ]:
df_long = pd.DataFrame({
    "Date": ["2023-01-01", "2023-01-01"],
    "Stock": ["AAPL", "AAPL"],
    "Price": [150, 155]
})

print(df_long)

# This will raise an error
# df_pivoted = df_long.pivot(index="Date", columns="Stock", values="Price")

# Note that we have 2023-01-01 twice for AAPL, which cannot work in wide form
# since we would have the same index twice, for the same ticker.

         Date Stock  Price
0  2023-01-01  AAPL    150
1  2023-01-01  AAPL    155


**Why df.pivot() Is Important**
1. Reorganizing Data for Analysis:

- Makes it easier to compare data across categories (e.g., stock prices by ticker).

2. Preparing Data for Visualization:

- Wide-format data is often required for plotting time-series data.

3. Simplifies Queries:

- You can easily access specific rows or columns after pivoting.


---



### 4. `pd.melt`
The pd.melt() method reshapes a DataFrame from wide format to long format by turning column headers into rows. This is particularly useful for preparing data for analysis, visualization, or storage in a database.

**What Does pd.melt() Do?**

- Converts multiple columns into a single column of values.
- Keeps specified columns (id_vars) unchanged as identifiers.
- Is the opposite of pivot or pivot_table.


**Parameters of pd.melt()**
1. frame:

 - The DataFrame to reshape.
2. id_vars:

 - Column(s) to keep as identifiers. These columns remain unchanged in the reshaped DataFrame.
3. value_vars:

 - Column(s) to unpivot (default: all columns except id_vars).

4. var_name:

 - Name for the new column containing the original column headers.

5. value_name:

 - Name for the new column containing the values from the unpivoted columns.

 Just remember that melt is how we convert to long from from wide form. This is very important since some machine learning algorithms and plotting techniques requires the data to be in long form.

 The examples will illustrate how to use the parameters.

In [ ]:
# Example 1: Basic melt
# Create a wide-format DataFrame
df_wide = pd.DataFrame({
    "Date": ["2023-01-01", "2023-01-02"],
    "AAPL": [150, 155],
    "MSFT": [250, 255]
})

print('\nThis is the wide form df\n')
print(df_wide)

# Melt the DataFrame
df_long = pd.melt(df_wide, id_vars=["Date"], var_name="Stock", value_name="Price")
print('\nThis is the long form df\n')
print(df_long)



This is the wide form df

         Date  AAPL  MSFT
0  2023-01-01   150   250
1  2023-01-02   155   255

This is the long form df

         Date Stock  Price
0  2023-01-01  AAPL    150
1  2023-01-02  AAPL    155
2  2023-01-01  MSFT    250
3  2023-01-02  MSFT    255


Let's look at the output so we can understand what is going on:

- the column labels of AAPL and MSFT must now be values in a new column, which we will name Stock. We tell Python that the new column will be called Stock via the `var_name` parameter.
- The data we want to show will be specified in the `value_name` parameter. In this case, we are saying call the column Price.
- And we had our dates in the Date column, and we want to keep that column also, but specify that it is a key identifier. Notice that it is supplied as a list.

Question: What if our dates were the index of df_wide and not a separate column. What would do?

Answer: reset the index first!

In [ ]:
# Example 2: Melt with default parameters
# Melt without specifying var_name or value_name
df_long_default = pd.melt(df_wide, id_vars=["Date"])
print(df_long_default)



         Date variable  value
0  2023-01-01     AAPL    150
1  2023-01-02     AAPL    155
2  2023-01-01     MSFT    250
3  2023-01-02     MSFT    255


In [ ]:
# Example 3: Melt a subset of columns
# You can specify which columns to unpivot using value_vars.

# Melt only the AAPL column
df_long_subset = pd.melt(df_wide, id_vars=["Date"], value_vars=["AAPL"], var_name="Stock", value_name="Price")
print(df_long_subset)

# This will only keep AAPL data


         Date Stock  Price
0  2023-01-01  AAPL    150
1  2023-01-02  AAPL    155


In [ ]:
# Example 4: Handling missing values
# If the original DataFrame contains NaN values, they are preserved in the melted DataFrame.

# Create a wide-format DataFrame with missing values
df_wide_missing = pd.DataFrame({
    "Date": ["2023-01-01", "2023-01-02"],
    "AAPL": [150, 155],
    "MSFT": [250, None]
})

print('\nThis is the wide form df\n')
print(df_wide_missing)

# Melt the DataFrame
df_long_missing = pd.melt(df_wide_missing, id_vars=["Date"], var_name="Stock", value_name="Price")
print('\nThis is the long form df\n')
print(df_long_missing)


This is the wide form df

         Date  AAPL   MSFT
0  2023-01-01   150  250.0
1  2023-01-02   155    NaN

This is the long form df

         Date Stock  Price
0  2023-01-01  AAPL  150.0
1  2023-01-02  AAPL  155.0
2  2023-01-01  MSFT  250.0
3  2023-01-02  MSFT    NaN


**Why pd.melt() Is Important**
1. Prepares Data for Visualization:

- Long format is often required by visualization libraries like seaborn and matplotlib.

2. Makes Data Analysis Easier:

- Grouping, filtering, and summarizing data is more straightforward in long format.

3. Standardizes Data Storage:

- Databases and statistical tools often prefer long format for flexibility and consistency.


---

### 5. `.pivot_table`

The `pd.pivot_table()` method is a powerful tool for summarizing and reshaping data. It allows you to create a wide-format table while applying aggregation functions (e.g., mean, sum) to handle duplicates or summarize data during the pivoting process.

This is the true equivalent of a pivot table in excel.

**What Does pd.pivot_table() Do?**
- Reshapes a DataFrame from long format to wide format.
- Handles duplicate combinations of rows and columns by applying an aggregation function.
- Provides additional flexibility compared to pivot().

**Parameters of pd.pivot_table()**
1. `data`:

- The DataFrame to pivot.
2. `index`:

- Specifies the column(s) to use as the new row index.
- Example: Use a Date column as the index.
3. `columns`:

- Specifies the column whose unique values will become new column headers.
- Example: Use a Stock column to create headers like AAPL, MSFT.

4. `values`:

- Specifies the column whose values will populate the table.
- Example: Use a Price column to populate the table with stock prices.

5. `aggfunc`:

- The aggregation function to apply to duplicate rows.
- Common options: mean (default), sum, max, min, count.

6. `fill_value`:

- Replaces missing values (NaN) with a specified value.

Pay special attention to the aggfunc!


In [ ]:
# Example 1: Basic pivot table
# Create a long-format DataFrame
df_long = pd.DataFrame({
    "Date": ["2023-01-01", "2023-01-01", "2023-01-02", "2023-01-02"],
    "Stock": ["AAPL", "MSFT", "AAPL", "MSFT"],
    "Price": [150, 250, 155, 255]
})

print('Long form before pivotting\n')
print(df_long)

# Pivot table with mean aggregation (default)
df_pivot = pd.pivot_table(df_long, index="Date", columns="Stock", values="Price", aggfunc="mean")
print('\nWide form after pivotting\n')
print(df_pivot)


Long form before pivotting

         Date Stock  Price
0  2023-01-01  AAPL    150
1  2023-01-01  MSFT    250
2  2023-01-02  AAPL    155
3  2023-01-02  MSFT    255

Wide form after pivotting

Stock        AAPL   MSFT
Date                    
2023-01-01  150.0  250.0
2023-01-02  155.0  255.0


Let's look at the above output:

- The Date column becomes the row index.
- The Stock column's unique values (AAPL, MSFT) become new column headers.
- The Price column populates the table, and since there are no duplicates, the result is the same as pivot(). This means that the aggfunc of mean was NOT used.

In [ ]:
# Example 2: Handling duplicate rows
# Now the aggfunc is used!

# Create a long-format DataFrame with duplicates
df_long_duplicates = pd.DataFrame({
    "Date": ["2023-01-01", "2023-01-01", "2023-01-02", "2023-01-02"],
    "Stock": ["AAPL", "AAPL", "MSFT", "MSFT"],
    "Price": [150, 155, 250, 255]
})

print('Long form before pivotting\n')
print(df_long_duplicates)

# Pivot table with mean aggregation
df_pivot_mean = pd.pivot_table(df_long_duplicates, index="Date", columns="Stock", values="Price", aggfunc="mean")
print('\n After pivotting and taking the mean of duplicates\n')
print(df_pivot_mean)


Long form before pivotting

         Date Stock  Price
0  2023-01-01  AAPL    150
1  2023-01-01  AAPL    155
2  2023-01-02  MSFT    250
3  2023-01-02  MSFT    255

 After pivotting and taking the mean of duplicates

Stock        AAPL   MSFT
Date                    
2023-01-01  152.5    NaN
2023-01-02    NaN  252.5


Explanation:

For AAPL on 2023-01-01, pd.pivot_table calculates the mean: (150 + 155) / 2 = 152.5.

If there are no values for a specific combination, NaN is shown unless you use fill_value. And that's what you see for the 2 missing values above. Why? There was no data in the long form for 2023-01-02 for AAPL!

In [ ]:
# Example 3: Adding a fill_value
# Pivot table with fill_value
df_pivot_fill = pd.pivot_table(df_long_duplicates, index="Date", columns="Stock", values="Price", fill_value=0)
print(df_pivot_fill)

# This produces the same table as above, but with no NaN.

Stock        AAPL   MSFT
Date                    
2023-01-01  152.5    0.0
2023-01-02    0.0  252.5


Explanation:

The fill_value=0 replaces NaN with 0, making the table easier to interpret.

In [ ]:
# Example 4: Using Different Aggregation Functions
# You can specify any aggregation function that works on grouped data.

# Pivot table with sum aggregation
df_pivot_sum = pd.pivot_table(df_long_duplicates, index="Date", columns="Stock", values="Price", aggfunc="sum")
print(df_pivot_sum)


Stock        AAPL   MSFT
Date                    
2023-01-01  305.0    NaN
2023-01-02    NaN  505.0


**Why pd.pivot_table() Is Important**

1. Handles Duplicate Entries:

 - Unlike pivot(), pivot_table handles duplicate index-column combinations with aggregation.
2. Data Summarization:

 - Provides a flexible way to group and summarize data using various aggregation functions.
3. Dealing with Missing Values:

 - Options like fill_value allow you to control how missing data is handled.



---

## Exporting and Importing Data

Pandas comes loaded with built in methods to import your data as a dataframe, and of course, export dataframes to multiple different file types.

At the moment, we will use just CSV and PKL files.

We will keep this very simple and use pricing data for SPY as our data set.

In [ ]:
# Download prices for SPY
df_spy = yf.download('SPY', start_date, end_date)

# Remove the multi index
df_spy.columns = df_spy.columns.get_level_values(0)

df_spy.head()


[*********************100%***********************]  1 of 1 completed


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,82.842194,120.300003,121.760002,119.900002,121.559998,55748000
2005-01-04,81.829903,118.830002,120.540001,118.440002,120.459999,69167600
2005-01-05,81.265228,118.010002,119.250000,118.000000,118.739998,65667300
2005-01-06,81.678391,118.610001,119.150002,118.260002,118.440002,47814700
2005-01-07,81.561348,118.440002,119.230003,118.129997,118.970001,55847700


### Exporting to a CSV file.

```
df_spy.to_csv(
    "spy_data.csv",   # File name to save

    index=True,       # Include the index column

    header=True,      # Include column headers

    sep=",",          # Specify the delimiter (default is a comma)

    encoding="utf-8", # Set file encoding

    float_format="%.2f" # Format for floating-point numbers

)
```



The above code is there to just demonstrate some of the optional parameters.

In [ ]:
# Export the dataframe to a CSV
# Normally, you are fine with just:
df_spy.to_csv('spy_prices.csv')  # just make sure you don't forger the .csv!

# If you look at the file explorer to our left in Colab, you will now see the file.

Now let's import it back in. Take a look at the key parameters that should be specified. Also note the versatility of this and how much of the heavy lifting Pandas can do for you.

```
df_spy_imported = pd.read_csv(
    "spy_data.csv",   # File path or name
    index_col="Date", # Specify which column to use as the index
    header=0,         # Specify the row number of the column headers (default is 0)
    parse_dates=True, # Convert date strings into datetime objects
    na_values=["NA", "NaN"], # Define which values should be interpreted as NaN
    keep_default_na=True # Include Pandas' default NaN identifiers
)
```
What is really cool, is that you can tell Python how NaN are represented in your spreadsheet! It can be #N/A (the default Excel NaN) or even blanks " ".

In [ ]:
# Note that any files you import into Colab must be found in the file explorer to your left!
# You can drag and drop any files from your local into Colab.
# Just note that those files are only here as long as Colab is running. Once your session ends
# the file system is gone. Make sure to download any files you exported!

df_spy_imported = pd.read_csv(
    "spy_prices.csv",   # File path or name
    index_col="Date", # Specify which column to use as the index
    header=0,         # Specify the row number of the column headers (default is 0)
    parse_dates=True, # Convert date strings into datetime objects
    na_values=["NA", "NaN"], # Define which values should be interpreted as NaN
    keep_default_na=True # Include Pandas' default NaN identifiers
)

df_spy_imported.head()

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,82.842194,120.300003,121.760002,119.900002,121.559998,55748000
2005-01-04,81.829903,118.830002,120.540001,118.440002,120.459999,69167600
2005-01-05,81.265228,118.010002,119.250000,118.000000,118.739998,65667300
2005-01-06,81.678391,118.610001,119.150002,118.260002,118.440002,47814700
2005-01-07,81.561348,118.440002,119.230003,118.129997,118.970001,55847700


And we have our data imported! Before we move on, we should check to see what format our index is in. We want it in datetime.

In [ ]:
df_spy_imported.index

DatetimeIndex(['2005-01-03', '2005-01-04', '2005-01-05', '2005-01-06',
               '2005-01-07', '2005-01-10', '2005-01-11', '2005-01-12',
               '2005-01-13', '2005-01-14',
               ...
               '2024-11-15', '2024-11-18', '2024-11-19', '2024-11-20',
               '2024-11-21', '2024-11-22', '2024-11-25', '2024-11-26',
               '2024-11-27', '2024-11-29'],
              dtype='datetime64[ns]', name='Date', length=5012, freq=None)

And it is! This is because we set `parse_dates` to True.

### Exporting to a PKL file
A .pkl file, short for Pickle file, is a serialized file format used by Python to save objects in a binary form. In the context of Pandas, it is used to store DataFrames or Series efficiently.

**Key Characteristics of .pkl Files**
1. Serialization:

- Pickling is the process of converting a Python object into a binary format that can be saved to a file or transferred over a network.

2. Preserves Structure:

- Unlike .csv, .pkl files retain the full structure and metadata of a DataFrame (e.g., data types, index, and column labels), making it ideal for saving complex data.

3. Fast Read/Write:

- .pkl files are faster to read and write compared to .csv files because they don’t involve converting the data to text.

4. Python-Specific:

- Pickle files are specific to Python, so they are not readable by non-Python tools.


**Why Use .pkl?**

- Speed and Efficiency: Faster and smaller file sizes compared to .csv for large datasets.
- Complex Objects: Can store DataFrames with nested structures, timestamps, or other non-tabular data.
- Quick Prototyping: Ideal for intermediate steps in a workflow where data needs to be saved and reloaded without loss of structure.

**When Not to Use .pkl**
- Sharing Across Platforms: Use .csv or other universal formats if the file needs to be used outside Python.
- Long-Term Storage: .pkl files may not be future-proof, as changes in Python or Pandas versions could affect compatibility.


In [ ]:
# Save our data as a pkl file
df_spy.to_pickle('spy_prices.pkl')

# Notice that we don't to specify any of parameters about index and header!
# It's all saved in the file format.

In [ ]:
df_spy_imported = pd.read_pickle("spy_prices.pkl")
df_spy_imported.head()

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,82.842194,120.300003,121.760002,119.900002,121.559998,55748000
2005-01-04,81.829903,118.830002,120.540001,118.440002,120.459999,69167600
2005-01-05,81.265228,118.010002,119.250000,118.000000,118.739998,65667300
2005-01-06,81.678391,118.610001,119.150002,118.260002,118.440002,47814700
2005-01-07,81.561348,118.440002,119.230003,118.129997,118.970001,55847700


And that's it! You now know how to save and import data, which is really important when dealing with large datasets.

## Conclusion

**Congratulations!**
You are now a pandas expert. More importantly, you've learned what you will need to succeed as financial data scientist as everything you learned is focused on specific use cases for financial data.

yes, there is a lot we didn't cover. And we will cover it in subsequent courses. But for now, the most important part is getting used to time series data, manipulating the dataframes, and learning how to slide and dice.

You are now ready for part 4 in this series, which will focus on Python functions and some other advanced concepts.

And make sure to try all of the sample problems associated with this section. It will help to reinforce what you learnt, and test your understanding of the concepts.